# Two-quantum-dot Rabi model transport (strong coupling) — refactored

Sectioned reorganization of `Dot_Class.ipynb`. One class per cell.
Broken cells (`SystemParameters` / `MasterEquationBuilder` /
`TransitionRateMatrix` / `*0` / `*withcavity` variants) have been
removed. Sweep figures are produced by a single `iv_curve_sweep`
helper.


## 0. Imports


In [ ]:
import numpy as np
import sympy as sp

from scipy.linalg import eigh
from scipy.special import eval_genlaguerre, factorial
import matplotlib.pyplot as plt

from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
import copy
import os

# Directory for saving figures
PLOT_DIR = "figures"
os.makedirs(PLOT_DIR, exist_ok=True)


## 1. Parameter dataclasses


### 1a. `DotParameters` — dot energies and inter-dot hopping


In [ ]:
class DotParameters:
    def __init__(self, e1, e2, t):
        
        self.e1 = e1
        self.e2 = e2
        self.t = t


### 1b. `CavityParameters` — photon cutoff, frequency, decay rate κ


In [ ]:
class CavityParameters:
    def __init__(self, n, omega, kappa):
        
        self.n = n
        self.omega = omega
        self.kappa = kappa


### 1c. `LeadParameters` — chemical potentials, temperatures, lead-dressing λ_L/R


In [ ]:
class LeadParameters:
    def __init__(self, gamma_L, gamma_R, mu_L, mu_R, mu_0, T_L, T_R, lam_L, lam_R):
        
        self.gamma_L = gamma_L
        self.gamma_R = gamma_R
        self.mu_L = mu_L
        self.mu_R = mu_R
        self.mu_0 = mu_0
        self.T_L = T_L
        self.T_R = T_R
        self.lam_L = lam_L
        self.lam_R = lam_R
        
    # -------------------------------------------------
    # Fermi function belongs naturally to the lead
    # -------------------------------------------------

    def fermi(self, E, lead="L"):
        """
        Fermi distribution for left or right lead.
        """
        E = np.array(E)
        if lead == "L":
            mu = self.mu_L
            T = self.T_L
        else:
            mu = self.mu_R
            T = self.T_R
        # Zero temperature
        if T == 0:
            return np.where(E < mu, 1.0, 0.0)
        # Finite temperature
        x = (E - mu) / T
        x = np.clip(x, -700, 700)  # prevent overflow
        return 1.0 / (np.exp(x) + 1.0)


## 2. Operators


### 2a. `PhotonicOperators` — a, a†, n̂ in cavity Fock basis


In [ ]:
class PhotonicOperators:
    def __init__(self, cavity: CavityParameters):
        """
        Constructs photonic operators a and a^\dagger
        in the combined charge × photon Hilbert space.

        Parameters
        ----------
        n_photons : int
            Maximum photon number (truncated at n_photons),
            giving photon space dimension n_photons + 1.
        charge_dim : int
            Total dimension of the charge space (default 4).
        """
        self.cavity = cavity
        self.n = self.cavity.n
        self.photon_dim = self.n + 1

        # Identity in charge space
        self.I_charge = np.eye(4)

        # Photon operators in photon space
        self.a_photon = self._create_a_photon()
        self.adag_photon = self._create_adag_photon()

        # Lift to total charge × photon space
        self.a = np.kron(self.I_charge, self.a_photon)
        self.adag = np.kron(self.I_charge, self.adag_photon)

    def _create_a_photon(self):
        """Photon annihilation operator in truncated Fock basis."""
        a = np.zeros((self.photon_dim, self.photon_dim))
        for m in range(1, self.photon_dim):
            a[m-1, m] = np.sqrt(m)
        return a

    def _create_adag_photon(self):
        """Photon creation operator in truncated Fock basis."""
        adag = np.zeros((self.photon_dim, self.photon_dim))
        for m in range(self.photon_dim - 1):
            adag[m+1, m] = np.sqrt(m+1)
        return adag
        


### 2b. `ElectronOperators` — d_L, d_R in the 4-state dot basis


In [ ]:
class ElectronOperators:
    """
    4×4 charge operators per phonon number:
    basis = |0>, |L>, |R>, |2>
    """
    
    @staticmethod
    def annihilation():
        d_l = np.zeros((4, 4), dtype=np.complex128)
        d_r = np.zeros((4, 4), dtype=np.complex128)
    
        d_l[0, 1] = 1.0     # <0| d_L |L>
        d_l[2, 3] = 1.0     # <R| d_L |2>
    
        d_r[0, 2] = 1.0     # <0| d_R |R>
        d_r[1, 3] = -1.0    # <L| d_R |2>
    
        return d_l, d_r


### 2c. `TunnelOperators` — dressed tunneling V_ℓ = D̂(λ_ℓ) ⊗ d_ℓ


In [ ]:
class TunnelOperators:
    """
    Construct full tunneling operators for left and right leads
    including displacement (electron-photon) operators.
    """
    def __init__(self, cavity: CavityParameters, leads: LeadParameters):
        """
        Args:
            cavity: CavityParameters object (defines phonon cutoff)
            lead_params: LeadParameters object (contains lam_L, lam_R)
        """
        self.cavity = cavity
        self.leads = leads
        self.M = self.cavity.n + 1  # phonon dimension
        self.lam_L = leads.lam_L
        self.lam_R = leads.lam_R

    def build(self):
        """
        Returns:
            V_l_ann, V_r_ann, V_l_cre, V_r_cre 
            Full tunneling operators including phonons.
        """
        d_l, d_r = ElectronOperators.annihilation()

        # Displacement operators (phonon part)
        D_L = DisplacementOperator.matrix(self.cavity.n, self.leads.lam_L)
        D_R = DisplacementOperator.matrix(self.cavity.n, self.leads.lam_R)

        # Full operators in combined Hilbert space
        V_l_ann = np.kron(d_l, D_L)
        V_r_ann = np.kron(d_r, D_R)

        # Creation operators are Hermitian transpose
        V_l_cre = V_l_ann.T.conj()
        V_r_cre = V_r_ann.T.conj()

        return V_l_ann, V_r_ann, V_l_cre, V_r_cre


### 2d. `DisplacementOperator` — D̂(λ) = exp[λ(a − a†)] matrix elements


In [ ]:
class DisplacementOperator:
    @staticmethod
    def element(m, n, lam):
        s = np.exp(-lam**2 / 2)
        if m >= n:
            k = m - n
            return (
                s
                * np.sqrt(factorial(n) / factorial(m))
                * lam**k
                * eval_genlaguerre(n, k, lam**2)
            )
        else:
            k = n - m
            return (
                s
                * np.sqrt(factorial(m) / factorial(n))
                * (-lam)**k
                * eval_genlaguerre(m, k, lam**2)
            )
    @staticmethod
    def matrix(N, lam):
        return np.array(
            [[DisplacementOperator.element(m, n, lam) for n in range(N + 1)]
             for m in range(N + 1)],
            dtype=complex
        )


## 3. Hamiltonian

`HamiltonianBuilder` constructs H = H_dots + H_cav and diagonalizes per charge sector.


In [ ]:
class HamiltonianBuilder:
    def __init__(self, cavity: CavityParameters, dots: DotParameters, lam):
        self.cavity = cavity     # store the whole cavity object
        self.dots = dots
        self.lam = lam
        
        # phonon cutoff
        self.n = self.cavity.n
        self.M = self.n + 1
        self.dim = 4 * self.M
        
        # Build the full Hamiltonian once and store it
        self.H = self.build()
    
    """Now let's build the Hamiltonian step-by-step."""
    # -----------------------------
    # Sector builders
    # -----------------------------

    def _build_sector_0(self):
        """Constructs the zero-particle sector Hamiltonian."""
        H0 = np.zeros((self.M, self.M), dtype=np.complex128)
        for n in range(self.M):
            H0[n, n] = self.cavity.omega * n
        return H0

    def _build_sector_2(self):
        """Constructs the two-particle sector Hamiltonian."""
        H2 = np.zeros((self.M, self.M), dtype=np.complex128)
        for n in range(self.M):
            H2[n, n] = self.dots.e1 + self.dots.e2 + self.cavity.omega * n
        return H2

    def _build_sector_1(self):
        """Constructs the single-particle sector Hamiltonian."""
        H1 = np.zeros((2 * self.M, 2 * self.M), dtype=np.complex128)

        # L block
        for n in range(self.M):
            H1[n, n] = self.dots.e1 + self.cavity.omega * n

        # R block
        for n in range(self.M):
            H1[self.M + n, self.M + n] = self.dots.e2 + self.cavity.omega * n

        # Hopping
        for n in range(self.M):
            for m in range(self.M):
                iL = n
                iR = self.M + m

                H1[iL, iR] = (
                    self.dots.t
                    * DisplacementOperator.element(m, n, self.lam)
                )
                H1[iR, iL] = (
                    self.dots.t
                    * DisplacementOperator.element(n, m, -self.lam)
                )

        return (H1 + H1.conj().T) / 2.0

    # -----------------------------
    # Full Hamiltonian
    # -----------------------------

    def build(self):
        H0 = self._build_sector_0()
        H1 = self._build_sector_1()
        H2 = self._build_sector_2()

        H = np.zeros((self.dim, self.dim), dtype=np.complex128)

        idx0 = list(range(self.M))
        idx1 = list(range(self.M, self.M + 2 * self.M))
        idx2 = list(range(self.M + 2 * self.M, self.M + 3 * self.M))

        # Fill blocks
        H[np.ix_(idx0, idx0)] = H0
        H[np.ix_(idx1, idx1)] = H1
        H[np.ix_(idx2, idx2)] = H2

        return (H + H.conj().T) / 2.0

    # -----------------------------
    # Diagonalization
    # -----------------------------

    def diagonalize(self):
        """
        Diagonalize Hamiltonian and return eigenvalues/eigenvectors
        sorted by charge sector (0 → 1 → 2), and energy-sorted inside each sector).
        """
    
        eigvals, eigvecs = eigh(self.H)
    
        # --- Build charge operator ---
        Q = np.zeros((self.dim, self.dim))
        M = self.M
    
        Q[0:M, 0:M] = 0
        Q[M:3*M, M:3*M] = np.eye(2*M) * 1
        Q[3*M:4*M, 3*M:4*M] = np.eye(M) * 2
    
        # --- Compute charge expectation values ---
        charge_vals = [
            np.real(v.conj().T @ (Q @ v))
            for v in eigvecs.T
        ]
    
        # --- Collect indices by sector ---
        sector_indices = {0: [], 1: [], 2: []}
        tol = 1e-8
    
        for i, q in enumerate(charge_vals):
            for sector in [0, 1, 2]:
                if abs(q - sector) < tol:
                    sector_indices[sector].append(i)
                    break
    
        # --- Sort each sector internally by energy ---
        for sector in sector_indices:
            sector_indices[sector].sort(key=lambda i: eigvals[i])
    
        ordered_indices = (
            sector_indices[0]
            + sector_indices[1]
            + sector_indices[2]
        )
    
        eigvals_sorted = eigvals[ordered_indices]
        eigvecs_sorted = eigvecs[:, ordered_indices]
    
        state_charge = (
            [0]*len(sector_indices[0]) +
            [1]*len(sector_indices[1]) +
            [2]*len(sector_indices[2])
        )
    
        return eigvals_sorted, eigvecs_sorted, state_charge


## 4. Master equation


### 4a. `CavityDecayMatrix` — Lindblad photon-decay rate matrix Γ_ph


In [ ]:
class CavityDecayMatrix:

    def __init__(self, cavity, eigvals, eigvecs):
        self.cavity = cavity
        self.eigvals = eigvals
        self.eigvecs = eigvecs

        self.M = cavity.n + 1
        self.dim = 4 * self.M
        self.kappa = cavity.kappa

        self._transform_ph_operators()

    def _transform_ph_operators(self):

        ph_op = PhotonicOperators(self.cavity)

        Udag = self.eigvecs.conj().T
        U = self.eigvecs

        self.a = Udag @ ph_op.a @ U
        self.adag = Udag @ ph_op.adag @ U

    def _rate(self, i, f):

        Ei = self.eigvals[i]
        Ef = self.eigvals[f]

        # photon decay: must lower energy
        if Ef >= Ei:
            return 0.0

        W = self.a[f, i]
        return (abs(W)**2)*self.kappa

    def build(self):

        # Store as Γ_ph[i, f] = rate(i → f), the SAME [initial, final] convention
        # as TunnelRateMatrix, so RateEquationSolver (which does L = Γ_full.T) treats
        # photon decay and lead tunneling consistently. (Storing [f, i] here would
        # transpose only the decay term, turning emission into absorption and pumping
        # the cavity up — a bug that is invisible at κ→0 but dominates at finite κ.)
        Γ_ph = np.zeros((self.dim, self.dim))

        for i in range(self.dim):
            for f in range(self.dim):
                Γ_ph[i, f] = self._rate(i, f)

        return Γ_ph


### 4b. `TunnelRateMatrix` — Fermi-dressed lead tunneling rates Γ_L, Γ_R


In [ ]:
class TunnelRateMatrix:
    """
    Builds Γ_L, Γ_R, Γ_total in the eigenbasis,
    including full tunneling matrix elements.
    """

    def __init__(
        self,
        cavity: CavityParameters,
        dots: DotParameters,
        leads: LeadParameters,
        eigvals,
        eigvecs
        ):

        self.cavity = cavity
        self.dots = dots
        self.leads = leads

        self.eigvals = eigvals
        self.eigvecs = eigvecs

        # Hilbert space sizes
        self.M = self.cavity.n + 1
        self.dim = 4 * self.M

        # Sector index ranges (guaranteed ordered)
        self.idx0 = range(0, self.M)
        self.idx1 = range(self.M, 3 * self.M)
        self.idx2 = range(3 * self.M, 4 * self.M)

        # -----------------------------
        # Tunnel operators
        # -----------------------------
        
        self._transform_tunneling_operators()

    # -------------------------------------------------
    # Build and rotate tunneling operators
    # -------------------------------------------------
    def _transform_tunneling_operators(self):

        tunnel = TunnelOperators(self.cavity, self.leads)
        V_l_ann, V_r_ann, V_l_cre, V_r_cre = tunnel.build()

        # Transform to eigenbasis
        Udag = self.eigvecs.conj().T
        U = self.eigvecs

        self.V_l_ann = Udag @ V_l_ann @ U
        self.V_r_ann = Udag @ V_r_ann @ U
        self.V_l_cre = Udag @ V_l_cre @ U
        self.V_r_cre = Udag @ V_r_cre @ U

    # -------------------------------------------------
    # Single transition rate
    # -------------------------------------------------
    def _rate(self, i, f):
        """
        Compute Γ_L and Γ_R for transition i → f.
        """

        Ei = self.eigvals[i]
        Ef = self.eigvals[f]
        ΔE = Ef - Ei

        # Determine if particle is added or removed
        # (sector ordering guarantees this works)
        if i in self.idx0 and f in self.idx1:
            adding = True
        elif i in self.idx1 and f in self.idx2:
            adding = True
        elif i in self.idx1 and f in self.idx0:
            adding = False
        elif i in self.idx2 and f in self.idx1:
            adding = False
        else:
            return 0.0, 0.0  # forbidden transition

        if adding:
            WL = self.V_l_cre[f, i]
            WR = self.V_r_cre[f, i]

            fL = self.leads.fermi(ΔE, lead="L")
            fR = self.leads.fermi(ΔE, lead="R")

        else:
            WL = self.V_l_ann[f, i]
            WR = self.V_r_ann[f, i]

            # Note: ΔE = Ef - Ei < 0 for removal
            fL = 1 - self.leads.fermi(-ΔE, lead="L")
            fR = 1 - self.leads.fermi(-ΔE, lead="R")

        γL = self.leads.gamma_L
        γR = self.leads.gamma_R

        rL = γL * abs(WL) ** 2 * fL if WL != 0 else 0.0
        rR = γR * abs(WR) ** 2 * fR if WR != 0 else 0.0

        return rL, rR

    # -------------------------------------------------
    # Build full Γ matrices
    # -------------------------------------------------
    def build(self):

        Γ_L = np.zeros((self.dim, self.dim))
        Γ_R = np.zeros((self.dim, self.dim))

        # -------- 0 ↔ 1 transitions ----------
        for i in self.idx0:
            for f in self.idx1:
                rL, rR = self._rate(i, f)
                Γ_L[i, f] = rL
                Γ_R[i, f] = rR

                rL, rR = self._rate(f, i)
                Γ_L[f, i] = rL
                Γ_R[f, i] = rR

        # -------- 1 ↔ 2 transitions ----------
        for i in self.idx1:
            for f in self.idx2:
                rL, rR = self._rate(i, f)
                Γ_L[i, f] = rL
                Γ_R[i, f] = rR

                rL, rR = self._rate(f, i)
                Γ_L[f, i] = rL
                Γ_R[f, i] = rR

        Γ_total = Γ_L + Γ_R

        return Γ_L, Γ_R, Γ_total


### 4c. `RateEquationSolver` — steady-state population of eigenstates


In [ ]:
class RateEquationSolver:
    """
    Builds and solves the classical master equation

        dP/dt = L P

    from the total transition matrix Γ_tot.

    Keeps exact sector block structure:

           [ A   B   0 ]
    L  =   [ D   E   F ]
           [ 0   H   I ]
    """

    def __init__(self, Γ_tot, Γ_ph, cavity: CavityParameters):
        self.Γ_tot = Γ_tot
        self.Γ_ph = Γ_ph
        self.cavity = cavity
        self.Γ_full = self.Γ_tot + self.Γ_ph

        self.m = self.cavity.n + 1

        # sector sizes
        self.n0 = self.m
        self.n1 = 2 * self.m
        self.n2 = self.m

        self.dim = self.n0 + self.n1 + self.n2

        # index slices
        self.P0 = slice(0, self.n0)
        self.P1 = slice(self.n0, self.n0 + self.n1)
        self.P2 = slice(self.n0 + self.n1, self.dim)

        # build master matrix immediately
        self.L, self.blocks = self._build_master()

    # -------------------------------------------------
    # Build master equation matrix
    # -------------------------------------------------
    def _build_master(self):
        """
        Construct master equation matrix L from Γ_tot.
        """

        L = np.zeros((self.dim, self.dim), dtype=float)

        # --------------------------------------------------------- 
        
        """
        note that the rate i → f is given by Γ_{if} = rate (i,f) but in the rate 
        equation writes:
        
            dP_i/dt = - (L P)_i + L_{ii} P_i
                    = - \sum_f L_{if} P_f + L_{ii} P_i

        by comparing one gets L = Γ.T such that

            L_{if} = Γ_{fi}            for off-diagonal
            L_{ii} = \sum_f Γ_{if}     for diagonal
        
        """
        
        # Incoming rates: L_{fi} = Γ(i → f)
        # ---------------------------------------------------------
        
        L[:, :] = self.Γ_full.T

        # ---------------------------------------------------------
        # Fix diagonal elements:
        # L[i,i] = - sum_{f ≠ i} Γ(i → f)
        # ---------------------------------------------------------
        for i in range(self.dim):
            outgoing = np.sum(self.Γ_full.T[:, i])
            L[i, i] = -outgoing

        # ---------------------------------------------------------
        # Extract blocks (exact sector block structure)
        # ---------------------------------------------------------
        A = L[self.P0, self.P0]
        B = L[self.P0, self.P1]
        D = L[self.P1, self.P0]
        E = L[self.P1, self.P1]
        F = L[self.P1, self.P2]
        H = L[self.P2, self.P1]
        I = L[self.P2, self.P2]

        blocks = {
            "A": A, "B": B, "D": D,
            "E": E, "F": F,
            "H": H, "I": I
        }

        return L, blocks

    # -------------------------------------------------
    # Solve steady state
    # -------------------------------------------------
    def steady_solver(self):
        evals, evecs = np.linalg.eig(self.L)
        idx = np.argmin(np.abs(evals))
        P = np.real(evecs[:, idx])
        P /= np.sum(P)
        return P


## 5. Transport observables


### 5a. `TransportCalculator` — current I from steady-state populations


In [ ]:
class TransportCalculator:
    """
    Computes transport observables (currently current)
    from sector-ordered rate matrices and steady-state probabilities.
    """

    def __init__(self, cavity: CavityParameters, leads: LeadParameters):
        self.cavity = cavity
        self.leads = leads

        self.n_ph = cavity.n
        self.n0 = self.n_ph + 1
        self.n1 = 2 * (self.n_ph + 1)
        self.n2 = self.n_ph + 1
        self.dim = self.n0 + self.n1 + self.n2

        # sector slices (consistent everywhere in your codebase)
        self.i0 = slice(0, self.n0)
        self.i1 = slice(self.n0, self.n0 + self.n1)
        self.i2 = slice(self.n0 + self.n1, self.dim)

    # -------------------------------------------------
    # Current from one lead
    # -------------------------------------------------
    def compute_current(self, Γ_alpha, P):
        """
        Compute current J^(alpha) from lead alpha.

        Parameters
        ----------
        Γ_alpha : ndarray (dim x dim)
            Transition-rate matrix for one lead (Γ_L or Γ_R),
            with Γ[f,i] = rate i -> f.

        P : ndarray (dim,)
            Steady-state probability vector.

        Returns
        -------
        J : float
            Current from lead alpha.
        """

        if Γ_alpha.shape != (self.dim, self.dim):
            raise ValueError("Gamma matrix has wrong dimension.")
        if P.shape[0] != self.dim:
            raise ValueError("Probability vector has wrong dimension.")

        # Sector probability vectors
        P0 = P[self.i0]
        P1 = P[self.i1]
        P2 = P[self.i2]

        # Extract sector blocks
        G01 = Γ_alpha[self.i0, self.i1]   # 0 -> 1
        G10 = Γ_alpha[self.i1, self.i0]   # 1 -> 0
        G12 = Γ_alpha[self.i1, self.i2]   # 1 -> 2
        G21 = Γ_alpha[self.i2, self.i1]   # 2 -> 1

        # -------------------------------------------------
        # Vectorized sector-block current formula
        # -------------------------------------------------

        # term1 = sum_i P0[i] * sum_j Γ(i→j in 1)
        term1 = float(P0.dot(G01.sum(axis=1)))

        # term2 = sum_j P1[j] * sum_i (Γ(j→i in 2) - Γ(j→i in 0))
        term2 = float(P1.dot((G12 - G10).sum(axis=1)))

        # term3 = sum_i P2[i] * sum_j Γ(i→j in 1)
        term3 = float(P2.dot(G21.sum(axis=1)))

        # Symmetric normalization (your convention)
        γL = self.leads.gamma_L
        γR = self.leads.gamma_R
        norm = γL * γR / (γL + γR)

        J = (term1 + term2 - term3) / norm

        return J


## 6. Rabi strong coupling

Build and analyse the strong-coupling **Rabi** spectrum. §6.1 shows the Rabi
eigenvalues vs λ (parity ladders and polarons); §6.2 takes the **deep-strong**
($\lambda\gg1$) limit, where the single-charge spectrum collapses onto an
equally-spaced harmonic ladder of doublets. The transport in this same limit is examined in §7, and the comparison against the
weak-coupling **Jaynes–Cummings** model in §8.

### 6.1. Rabi spectrum vs λ

Diagonalise the full polaron-transformed Rabi Hamiltonian (`HamiltonianBuilder`) over `λ ∈ [0, 3]` for the bright-bonding setup and plot all eigenvalues, colour-coded by charge sector (0 / 1 / 2). Sectors 0 and 2 are flat (only photonic, `ω n` and `ε₁+ε₂+ω n`); the polariton sector 1 shows the λ-driven avoided crossings. A second plot then resolves the single-charge sector by **parity**, exposing the two ladders and the Rabi doublets (below).


In [ ]:
dots = DotParameters(e1=0.6, e2=0.6, t=0.5)
cavity = CavityParameters(n=8, omega=1.0, kappa=1e-3)
M = cavity.n + 1
dim = 4 * M

lam_values = np.linspace(0.0, 3.0, 300)

# Colour the single-charge sector (sector 1) by the hidden parity P=(1<->2)x(a->-a):
# EVEN parity (P=+1) red, ODD parity (P=-1) blue.  Within a parity, same-parity levels
# only avoid-cross, so energy-sorting each parity at every lambda gives smooth, non-crossing
# branches (opposite parities cross freely -> red and blue lines cross, as they should).
# Sectors 0 (empty) and 2 (2e+photons) are flat photonic ladders, kept gray solid / dashed.
phot_par = (-1.0) ** np.arange(M)                       # photon parity (-1)^m

sec0_E = np.empty((len(lam_values), M))
sec2_E = np.empty((len(lam_values), M))
even_E = np.empty((len(lam_values), M))                 # P = +1  (even)
odd_E  = np.empty((len(lam_values), M))                 # P = -1  (odd)

for k, lam in enumerate(lam_values):
    ev, evec, _ = HamiltonianBuilder(cavity, dots, lam).diagonalize()
    sec0_E[k] = np.sort(ev[0:M].real)
    sec2_E[k] = np.sort(ev[3 * M:4 * M].real)
    # parity expectation <psi|P|psi> = 2 Re sum_m (-1)^m conj(L_m) R_m for each sector-1 eigenvector
    L = evec[M:2 * M, M:3 * M]                           # |dot1,m> amplitudes (M x 2M)
    R = evec[2 * M:3 * M, M:3 * M]                       # |dot2,m> amplitudes
    v = 2.0 * np.real(np.sum(phot_par[:, None] * np.conj(L) * R, axis=0))   # length 2M
    order = np.argsort(v)                                # robust M/M split (handles crossings)
    E1 = ev[M:3 * M].real
    odd_E[k]  = np.sort(E1[order[:M]])                   # most negative v -> odd  (P=-1)
    even_E[k] = np.sort(E1[order[M:]])                   # most positive v -> even (P=+1)

EVEN_COLOR, ODD_COLOR, SEC02_COLOR = "tab:red", "tab:blue", "tab:gray"

fig, ax = plt.subplots(figsize=(8, 5))
for j in range(M):                                       # sectors 0 / 2: gray solid / dashed
    ax.plot(lam_values, sec0_E[:, j], color=SEC02_COLOR, ls="-",  lw=0.8, alpha=0.8)
    ax.plot(lam_values, sec2_E[:, j], color=SEC02_COLOR, ls="--", lw=0.8, alpha=0.8)
for j in range(M):                                       # sector 1: parity-coloured
    ax.plot(lam_values, even_E[:, j], color=EVEN_COLOR, ls="-", lw=1.1, alpha=0.95)
    ax.plot(lam_values, odd_E[:,  j], color=ODD_COLOR,  ls="-", lw=1.1, alpha=0.95)

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0], [0], color=EVEN_COLOR,  ls="-",  label=r"Sector 1, even parity $P=+1$"),
    Line2D([0], [0], color=ODD_COLOR,   ls="-",  label=r"Sector 1, odd parity $P=-1$"),
    Line2D([0], [0], color=SEC02_COLOR, ls="-",  label="Sector 0"),
    Line2D([0], [0], color=SEC02_COLOR, ls="--", label="Sector 2"),
], loc="upper left", fontsize=8)

ax.set_xlabel(r"$\lambda$")
ax.set_ylabel("Eigenvalue")
ax.set_title(rf"Rabi spectrum"
             rf"($e_1=e_2={dots.e1}$, $t={dots.t}$, $\omega={cavity.omega}$, $N={M-1}$)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "rabi_spectrum_all_sectors"+".png"), dpi=150, bbox_inches="tight"); plt.show()

### 6.1 — parity ladders and the Rabi doublets

Within the **single-charge** sector the spectrum hides a $\mathbb{Z}_2$ structure: $H$ commutes with the
**combined parity**
$$\hat P=(1\!\leftrightarrow\!2)\times(a\to-a)$$
(dot-swap $\times$ cavity-quadrature flip), under which the dressed hopping is invariant since
$\hat D(\lambda)\to\hat D(-\lambda)$. The bare states carry definite parity,
$$\hat P\,|g,m\rangle=(-1)^{m+1}|g,m\rangle,\qquad \hat P\,|e,m\rangle=(-1)^m|e,m\rangle,$$
so the basis splits into **two ladders that never mix**:

| sector | bare states |
|---|---|
| $P=-1$ | $\{|g,0\rangle,|e,1\rangle,|g,2\rangle,|e,3\rangle,\dots\}$ — g-even, e-odd |
| $P=+1$ | $\{|e,0\rangle,|g,1\rangle,|e,2\rangle,|g,3\rangle,\dots\}$ — e-even, g-odd |

Spectral signature: **same-parity levels repel** (avoided crossings = the **Rabi doublets**), while
**opposite-parity levels cross freely**. The plot below colours the single-charge eigenvalues by parity —
same-colour curves never touch, opposite-colour curves do.

**First Rabi doublet.** At $\lambda=0$, $|e,0\rangle$ and $|g,1\rangle$ (both $P=+1$) are degenerate at
$E=e+t=\varepsilon_g+\omega$ (the resonance $\Delta\omega=\omega-2t=0$). Turning on $\lambda$ splits them —
the **vacuum-Rabi splitting** — into a lower ($\approx|e,0\rangle$) and upper ($\approx|g,1\rangle$) polariton,
both in the $P=+1$ ladder.

**Strong-coupling caveat & labelling.** The clean $\tfrac1{\sqrt2}(|e,0\rangle\pm|g,1\rangle)$ form
holds only as $\lambda\to0$; by $\lambda=1$ the hopping is renormalised to $t_{\rm eff}=t\,e^{-\lambda^2/2}$,
detuning the manifold, so the lower polariton is $\sim$83% $|e,0\rangle$ (full decomposition below).
Crucially, **energy-ordering interleaves the two ladders**, so a naive energy-ordered $\pm$ would split
*non*-partners. We therefore label by **Rabi-partner manifold** (the `label_rabi_partners` helper below):
sort each parity ladder by energy, pair consecutive branches into its Jaynes–Cummings manifolds $N$, and
call the lower/upper member of manifold $n$ the partners $|n^-\rangle/|n^+\rangle$. Because both members
share parity, $|n^-\rangle,|n^+\rangle$ are the **genuine Rabi partners by construction** — on *and* off
resonance — even when an opposite-parity level sits between them in energy.


In [ ]:
# 6.1 — helper: label the single-charge eigenstates by Rabi-partner manifold |n^±>.
#
# The hidden parity P = (1<->2) x (a->-a) splits the single-charge sector into two ladders that
# never mix; same-parity levels only AVOID-cross, so each parity ladder is a set of non-crossing
# branches. Sorting a ladder by energy and pairing consecutive branches recovers the
# Jaynes-Cummings excitation manifolds N: the ladder holding the global ground |g~,0> carries the
# EVEN manifolds N=0,2,4,... (N=0 = the lone ground singlet); the other ladder carries the ODD
# manifolds N=1,3,5,.... Within a manifold, '-' = lower energy, '+' = upper. Because both members
# of a manifold share parity, |n^-> and |n^+> are the GENUINE Rabi partners -- on AND off
# resonance -- even when an opposite-parity level sits between them in energy.
#
# Parity-resolved ordinal scheme: robust at all lambda since there are no true crossings inside a
# parity ladder (so each (N,branch) is one smooth branch). At extreme coupling, where a ladder is
# heavily reshuffled, cross-check against adiabatic continuation from lambda=0.

def label_rabi_partners(eigvals, evec, Ms):
    """Map each single-charge eigenstate (columns Ms..3Ms-1) to its Rabi-partner label.

    Returns {column_index: dict(E, parity=+/-1, N, branch in {'0','-','+','top'},
             label (LaTeX), tag (ascii))}.
    """
    phot_par = np.array([(-1) ** m for m in range(Ms)])            # photon parity (-1)^m
    idx = list(range(Ms, 3 * Ms))                                  # single-charge columns

    def parity(k):                                                 # sign of <psi|P|psi>
        psi = evec[:, k]
        L, R = psi[Ms:2 * Ms], psi[2 * Ms:3 * Ms]                  # |dot1,m>, |dot2,m> amplitudes
        v = 2 * np.real(np.sum(phot_par * np.conj(L) * R))
        return 1.0 if v >= 0 else -1.0

    par = {k: parity(k) for k in idx}
    idx.sort(key=lambda k: eigvals[k].real)                        # by energy
    gp = par[idx[0]]                                               # parity of the ground ladder
    even = [k for k in idx if par[k] == gp]                        # N = 0,2,4,...
    odd  = [k for k in idx if par[k] != gp]                        # N = 1,3,5,...

    def rec(k, p, n, br):
        tag = {'0': '|g~,0>', 'top': f'|{n}(cut)>'}.get(br, f'|{n}{br}>')
        lab = {'0': r'$|\tilde g,0\rangle$',
               'top': rf'$|{n}^{{(cut)}}\rangle$'}.get(br, rf'$|{n}^{br}\rangle$')
        return dict(E=eigvals[k].real, parity=p, N=n, branch=br, label=lab, tag=tag)

    out = {even[0]: rec(even[0], gp, 0, '0')}                      # lone ground singlet
    for ladder, p, n0 in ((even[1:], gp, 2), (odd, -gp, 1)):       # pair up each ladder
        for j in range(0, len(ladder) - 1, 2):
            lo, hi = ladder[j], ladder[j + 1]; n = n0 + (j // 2) * 2
            out[lo] = rec(lo, p, n, '-'); out[hi] = rec(hi, p, n, '+')
        if len(ladder) % 2:                                        # leftover = truncation top singlet
            k = ladder[-1]; out[k] = rec(k, p, n0 + (len(ladder) - 1), 'top')
    return out


In [ ]:
# 6.1 — single-charge spectrum, parity-coloured, with explicit Rabi-partner labels |n^±>.
# Branches are grouped into JC manifolds by `label_rabi_partners`: |n^-> (lower) and |n^+> (upper)
# are the genuine Rabi partners -- SAME parity colour, SAME manifold index n -- even though an
# opposite-parity level can sit between them in energy.
from collections import defaultdict
from matplotlib.lines import Line2D

dots_s = DotParameters(e1=0.6, e2=0.6, t=0.5)
cav_s  = CavityParameters(n=8, omega=1.0, kappa=1e-3)
Ms = cav_s.n + 1
lam_s = np.linspace(0.0, 5.0, 240)

series = defaultdict(lambda: ([], []))     # (N, branch) -> (lam[], E[])
meta   = {}                                # (N, branch) -> last label record
for lam in lam_s:
    ev, evec, _ = HamiltonianBuilder(cav_s, dots_s, lam).diagonalize()
    for info in label_rabi_partners(ev, evec, Ms).values():
        key = (info['N'], info['branch'])
        series[key][0].append(lam); series[key][1].append(info['E'])
        meta[key] = info

def edge_labels(ax, series, meta, lam, ymax, ymin, nmax=4, gap=0.17):
    """Right-margin |n^±> labels for the low manifolds, vertically dodged with thin leaders."""
    ents = []
    for key, (xs, Es) in series.items():
        info = meta[key]
        if info['branch'] == 'top' or info['N'] > nmax:
            continue
        vis = [i for i, e in enumerate(Es) if ymin < e < ymax - 0.04]    # last in-window point
        if vis:
            i = vis[-1]
            ents.append([xs[i], Es[i], Es[i], info['label'],
                         "tab:red" if info['parity'] > 0 else "tab:blue"])
    ents.sort(key=lambda r: r[2])
    for j in range(1, len(ents)):                                        # greedy upward dodge
        if ents[j][2] - ents[j-1][2] < gap:
            ents[j][2] = ents[j-1][2] + gap
    xL = lam[-1]
    for x0, y0, yl, txt, col in ents:
        ax.annotate(txt, xy=(x0, y0), xytext=(xL + 0.28, yl), textcoords="data",
                    fontsize=8, color=col, va="center", clip_on=False,
                    arrowprops=dict(arrowstyle="-", color=col, lw=0.5, alpha=0.5),
                    bbox=dict(boxstyle="round,pad=0.08", fc="white", ec="none", alpha=0.75))

fig, ax = plt.subplots(figsize=(8.8, 6)); ymax, ymin = 4.0, -0.1
for key, (xs, Es) in series.items():
    col = "tab:red" if meta[key]['parity'] > 0 else "tab:blue"
    ax.plot(xs, Es, color=col, lw=1.1)
edge_labels(ax, series, meta, lam_s, ymax, ymin)

ax.legend(handles=[Line2D([0], [0], color="tab:red", lw=2, label=r"$P=+1$  (odd manifolds $N=1,3,\dots$)"),
                   Line2D([0], [0], color="tab:blue", lw=2, label=r"$P=-1$  (even manifolds $N=0,2,\dots$)")],
          loc="lower right", fontsize=9)
ax.set_xlabel(r"$\lambda$"); ax.set_ylabel("single-charge eigenvalue")
ax.set_xlim(lam_s[0], lam_s[-1] + 0.75); ax.set_ylim(ymin, ymax)
ax.set_title("Single-charge sector — Rabi-partner manifolds $|n^\\pm\\rangle$\n"
             "($|n^-\\rangle,|n^+\\rangle$ = same parity colour & manifold $n$; opposite-colour levels cross freely)")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "rabi_single_charge_spectrum_resonant"+".png"), dpi=150, bbox_inches="tight"); plt.show()


### 6.1 — off-resonance: the doublets when $\Delta\omega\neq0$

Parity $\hat P$ is a symmetry of $H$ for **any** parameters, so the two ladders survive off resonance; what
moves is *where* the degeneracies sit. Take $e_1=e_2=t=\omega=1$, i.e. $\varepsilon_g=e-t=0$,
$\varepsilon_e=e+t=2$, and $\Delta\omega=\omega-2t=-1$.

- **On resonance** ($\Delta\omega=0$, plot above): the same-parity pair $|e,0\rangle,|g,1\rangle$ is degenerate
  at $\lambda=0$ and splits immediately — the vacuum-Rabi doublet, with gap $\to0$ as $\lambda\to0$.
- **Off resonance** (here): **no same-parity pair is degenerate at $\lambda=0$.** The $\lambda=0$
  degeneracies ($E=2,3,4,\dots$) are all **opposite-parity** pairs — e.g. $|e,0\rangle\,(P{=}+1)$ and
  $|g,2\rangle\,(P{=}-1)$ at $E=2$ — which therefore **cross freely** as $\lambda$ turns on. The same-parity
  (Rabi) avoided crossings still occur, but only at **finite $\lambda$** and with a **finite minimum gap** set
  by the detuning — they *never close*. The tightest here is a red ($P{=}+1$) doublet with gap $\approx0.23$
  near $\lambda\approx1.06$, $E\approx2.5$.

The plot keeps the same two-colour parity scheme and marks one of each: a finite-gap same-parity Rabi doublet
and an opposite-parity free crossing.

In [ ]:
# 6.1 (off-resonance) — single-charge spectrum, parity-coloured, Rabi-partner labels |n^±>.
#   e1=e2=t=1, omega=1:  eps_g=0, eps_e=2, detuning Delta_omega=omega-2t=-1.
# Same manifold labelling (`label_rabi_partners`) as the resonant plot; here NO same-parity pair is
# degenerate at lambda=0, so every |n^->,|n^+> Rabi doublet opens only at finite lambda, finite gap.
from collections import defaultdict
from matplotlib.lines import Line2D

dots_o = DotParameters(e1=1, e2=1, t=1)
cav_o  = CavityParameters(n=8, omega=1.0, kappa=1e-3)
Mo = cav_o.n + 1
lam_o = np.linspace(0.0, 5.0, 240)

series = defaultdict(lambda: ([], [])); meta = {}
for lam in lam_o:
    ev, evec, _ = HamiltonianBuilder(cav_o, dots_o, lam).diagonalize()
    for info in label_rabi_partners(ev, evec, Mo).values():
        key = (info['N'], info['branch'])
        series[key][0].append(lam); series[key][1].append(info['E']); meta[key] = info

fig, ax = plt.subplots(figsize=(8.8, 6)); ymax, ymin = 5.0, -0.2
for key, (xs, Es) in series.items():
    col = "tab:red" if meta[key]['parity'] > 0 else "tab:blue"
    ax.plot(xs, Es, color=col, lw=1.1)
edge_labels(ax, series, meta, lam_o, ymax, ymin)        # helper defined in the resonant cell above

################
iref = int(np.argmin(np.abs(lam_o - 1.0))); xref = lam_o[iref]
E1p, E3m = series[(1, '+')][1][iref], series[(3, '-')][1][iref]
ax.plot([xref, xref], [E1p, E3m], color="tab:green", lw=1, alpha=1, linestyle=":", zorder=0.0)
################

# a finite-gap same-parity Rabi doublet (opens only at finite lambda off resonance)
ax.annotate("finite gap (anti-crossing)",
            xy=(1.06, 2.50), xytext=(1.5, 2.50), fontsize=8, color="0.25",
            arrowprops=dict(arrowstyle="->", color="0.25", lw=1.0))
# lambda=0 degeneracy at E=2 is OPPOSITE parity (different manifolds) -> free crossing
ax.annotate(" degeneracy (opposite parity)",
            xy=(0.0, 2.0), xytext=(0.1, 1.30), fontsize=8, color="0.25",
            arrowprops=dict(arrowstyle="->", color="0.25", lw=1.0))

ax.legend(handles=[Line2D([0], [0], color="tab:red", lw=2, label=r"$P=+1$  (odd manifolds)"),
                   Line2D([0], [0], color="tab:blue", lw=2, label=r"$P=-1$  (even manifolds)")],
          loc="lower right", fontsize=9)
ax.set_xlabel(r"$\lambda$"); ax.set_ylabel("single-charge eigenvalue")
ax.set_xlim(lam_o[0], lam_o[-1] + 0.75); ax.set_ylim(ymin, ymax)
ax.set_title("Single-charge sector OFF-RESONANCE ($\\Delta\\omega=\\omega-2t=-1$) — Rabi-partner manifolds $|n^\\pm\\rangle$\n"
             "(every $|n^-\\rangle,|n^+\\rangle$ doublet opens at finite $\\lambda$ with a finite gap)")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "rabi_single_charge_spectrum_offresonance"+".png"), dpi=150, bbox_inches="tight"); plt.show()


### 6.1 — spectrum structure: the low-lying polarons in the bare basis

The single-charge eigenstates are **polarons** — dressed superpositions of the bare products
$\{|g,m\rangle,|e,m\rangle\}$, organised by the parity $\hat P$ into the two ladders above. They are
*not* products because the hopping is **photon-number-changing**,
$$\langle 2,n'|\,\mathfrak t\,e^{\lambda(a-a^\dagger)}d_1^\dagger d_2\,|1,n\rangle=\mathfrak t\,\langle n'|\hat D(\lambda)|n\rangle\neq0\quad(n'\neq n).$$
In the $g/e$ basis the single-charge block reads
$$H_1=(e+\omega a^\dagger a)\,\mathbb 1
+\tfrac{\mathfrak t}{2}\underbrace{(\hat D+\hat D^\dagger)}_{2\cosh,\ \Delta n\ \text{even}}\big(|g\rangle\langle g|-|e\rangle\langle e|\big)
+\tfrac{\mathfrak t}{2}\underbrace{(\hat D^\dagger-\hat D)}_{-2\sinh,\ \Delta n\ \text{odd}}\big(|g\rangle\langle e|-|e\rangle\langle g|\big),$$
so the $\sinh$ (odd) term mixes $|g,0\rangle$ with $|e,1\rangle$ (order $\lambda$) and the $\cosh$ (even)
term with $|g,2\rangle$ (order $\lambda^2$); at $\lambda=0$ everything decouples and each eigenstate is an
exact product. (Terminology: the **polarons are the eigenstates**; $|g,m\rangle,|e,m\rangle$ are the bare
product basis — not polarons.)

**Low-lying polarons at $\lambda=1$ — weights and dominant bare contributors:**

| eigenstate | $E$ | $P$ | dominant bare content (weights) |
|---|---|---|---|
| $\lvert\tilde g,0\rangle$ | 0.217 | $-1$ | 0.956 $\lvert g,0\rangle$ + 0.039 $\lvert e,1\rangle$ + 0.005 $\lvert g,2\rangle$ |
| $\lvert 1^-\rangle$ | 0.730 | $+1$ | 0.828 $\lvert e,0\rangle$ + 0.144 $\lvert g,1\rangle$ + 0.023 $\lvert e,2\rangle$ |
| $\lvert 2^-\rangle$ | 1.545 | $-1$ | 0.890 $\lvert e,1\rangle$ + 0.053 $\lvert g,2\rangle$ + 0.029 $\lvert g,0\rangle$ |
| $\lvert 1^+\rangle$ | 1.675 | $+1$ | 0.817 $\lvert g,1\rangle$ + 0.162 $\lvert e,0\rangle$ + 0.012 $\lvert e,2\rangle$ |
| $\lvert 3^-\rangle$ | 2.437 | $+1$ | 0.937 $\lvert e,2\rangle$ + 0.020 $\lvert g,1\rangle$ + 0.015 $\lvert e,4\rangle$ |

The structure is now transparent:
- **Two parity ladders (now indexed by manifold).** $P{=}-1$ (even $N$): $\lvert\tilde g,0\rangle\,(\sim\!\lvert g,0\rangle),\ \lvert2^-\rangle\,(\sim\!\lvert e,1\rangle),\ \lvert2^+\rangle\,(\sim\!\lvert g,2\rangle),\dots$ ;  $P{=}+1$ (odd $N$): $\lvert1^-\rangle\,(\sim\!\lvert e,0\rangle),\ \lvert1^+\rangle\,(\sim\!\lvert g,1\rangle),\ \lvert3^-\rangle\,(\sim\!\lvert e,2\rangle),\dots$
- **The genuine vacuum-Rabi doublet is $\{\lvert1^-\rangle,\lvert1^+\rangle\}$** — both $P{=}+1$, the lower (83% $\lvert e,0\rangle$) and upper (82% $\lvert g,1\rangle$) mixtures of the *same* $\{\lvert e,0\rangle,\lvert g,1\rangle\}$ pair, so the manifold labelling gives them the **same index $n{=}1$ by construction**. The energy-ordered neighbour $\lvert2^-\rangle$ ($P{=}-1$, $\sim\!\lvert e,1\rangle$) belongs to a **different** manifold and merely *sits between them* in energy — with `label_rabi_partners` the $\lvert n^\pm\rangle$ labels are the Rabi partners on and off resonance.
- Each polaron is **dominated by one bare product** ($\ge$82%), with a few-percent cloud whose weights fall in powers of $\lambda^2$ — the hallmark of perturbative dressing. (Used in §12 for the Franck–Condon transport channels.)

The `polaron_decomp` cell below prints these five (energy-ordered, manifold-labelled), plus the $\lambda=0$ product limit; $\lvert2^+\rangle\,(\sim\!\lvert g,2\rangle)$ — the partner of $\lvert2^-\rangle$ — lies just above at $E\approx2.79$.

In [ ]:
# 6.1 — decomposition of the low-lying single-charge eigenstates (polarons) in the bare basis
# {|g,m>, |e,m>}.  Labels |n^±> come from `label_rabi_partners` (Rabi-partner manifolds), so the two
# members of each doublet carry the SAME n.  At lam=0 each eigenstate is an exact product; at lam=1
# it is a polaron dominated by one bare state plus a small cloud.  (e=0.6, t=0.5, N=10.)

def polaron_decomp(lam, nstate=5, ncomp=5, e=0.6, t=0.5, N=10):
    dots = DotParameters(e1=e, e2=e, t=t)
    cav  = CavityParameters(n=N, omega=1.0, kappa=1e-3)
    ev, evec, _ = HamiltonianBuilder(cav, dots, lam).diagonalize()
    Ms = N + 1
    lab = label_rabi_partners(ev, evec, Ms)
    order = sorted(lab, key=lambda k: ev[k].real)[:nstate]         # single-charge eigenstates by energy
    p0 = evec[:, order[0]]                                          # fix g/e from the ground orbital
    use_minus = abs((p0[Ms:2*Ms] - p0[2*Ms:3*Ms])[0]) >= abs((p0[Ms:2*Ms] + p0[2*Ms:3*Ms])[0])
    for k in order:
        psi = evec[:, k]; L, R = psi[Ms:2*Ms], psi[2*Ms:3*Ms]
        g  = (L - R) / np.sqrt(2) if use_minus else (L + R) / np.sqrt(2)
        eo = (L + R) / np.sqrt(2) if use_minus else (L - R) / np.sqrt(2)
        wg, we = np.abs(g)**2, np.abs(eo)**2
        info = lab[k]; sec = f"P={'+' if info['parity'] > 0 else '-'}1  (N={info['N']})"
        print(f"{info['tag']:>8}  E={ev[k].real:.4f}  {sec}")
        print(f"   {'m':>2} {'|<g,m|.>|^2':>13} {'|<e,m|.>|^2':>13}")
        for m in range(ncomp):
            print(f"   {m:>2} {wg[m]:>13.5f} {we[m]:>13.5f}")
        print()

print("=== lam = 0:  eigenstates are EXACT products ===\n")
polaron_decomp(0.0, nstate=1)
print("=== lam = 1:  low-lying polarons (ground + first two manifolds) ===\n")
polaron_decomp(1.0, nstate=5)


### 6.1 — detuned dots ($e_1\neq e_2$): the parity is broken

For asymmetric dots the combined parity $\hat P=(1\!\leftrightarrow\!2)\times(a\to-a)$ is **no longer a symmetry**.
Under the dot swap the onsite term transforms as $\varepsilon_1 n_1+\varepsilon_2 n_2\to\varepsilon_1 n_2+\varepsilon_2 n_1$,
which equals the original **only if $\varepsilon_1=\varepsilon_2$** (the energies are fixed to lab sites; only the
occupations swap). Splitting it,
$$\varepsilon_1 n_1+\varepsilon_2 n_2=\tfrac{\varepsilon_1+\varepsilon_2}{2}(n_1+n_2)+\tfrac{\varepsilon_1-\varepsilon_2}{2}(n_1-n_2),$$
the average is $\hat P$-even but the **detuning piece $\tfrac{\varepsilon_1-\varepsilon_2}{2}(n_1-n_2)$ is $\hat P$-odd**
($\hat P(n_1-n_2)\hat P^\dagger=-(n_1-n_2)$), so $[\hat P,H]\propto(\varepsilon_1-\varepsilon_2)\neq0$ (verified
$\lVert[\hat P,H]\rVert=|\varepsilon_1-\varepsilon_2|\sqrt{2M}$). The **hopping stays invariant** — which is exactly why
the parity must combine the swap with $a\to-a$: the latter sends $\hat D(\lambda)\to\hat D(-\lambda)$ so the dressed
hopping maps onto its own h.c.

The plot below ($e_1=0.6,\ e_2=0.7$, $\delta=e_1-e_2=-0.1$) colours each single-charge branch by the **parity
expectation $\langle\hat P\rangle$**, which is now a *continuous* number in $[-1,+1]$ rather than a sharp $\pm1$:

- Far from crossings the states are still **nearly parity-pure** (saturated red $\langle\hat P\rangle\!\approx\!+1$
  / blue $\approx\!-1$), because the mixing scale $\delta$ is small next to the hopping $t$.
- At the **opposite-parity crossings that were free at $e_1=e_2$**, the $\hat P$-odd detuning now couples the two
  ladders: the levels **avoid-cross**, swap character, and pass through $\langle\hat P\rangle\!\approx\!0$ (white/grey).
  With no exact symmetry left, *every* crossing is gapped; the formerly protected free crossings acquire a small gap
  $\propto|\delta|$.

In [ ]:
# 6.1 — detuned dots (e1 != e2): broken parity.  Single-charge spectrum vs lambda,
# coloured by the CONTINUOUS parity expectation <P> = 2 Re sum_m (-1)^m conj(L_m) R_m
# (no longer +-1 because P is not conserved when e1 != e2).
from matplotlib.collections import LineCollection

dots_d = DotParameters(e1=0.6, e2=0.7, t=0.5)        # detuning delta = e1 - e2 = -0.1
cav_d  = CavityParameters(n=8, omega=1.0, kappa=1e-3)
Md     = cav_d.n + 1
lam_d  = np.linspace(0.0, 4.0, 400)
phot_par = (-1.0) ** np.arange(Md)

E_br = np.empty((len(lam_d), 2 * Md))                 # energy-sorted single-charge branches
P_br = np.empty((len(lam_d), 2 * Md))                 # <P> along each branch (continuous)
for k, lam in enumerate(lam_d):
    ev, evec, _ = HamiltonianBuilder(cav_d, dots_d, lam).diagonalize()
    E1 = ev[Md:3 * Md].real
    L = evec[Md:2 * Md, Md:3 * Md]; R = evec[2 * Md:3 * Md, Md:3 * Md]
    Pexp = 2.0 * np.real(np.sum(phot_par[:, None] * np.conj(L) * R, axis=0))   # in [-1, 1]
    order = np.argsort(E1)
    E_br[k] = E1[order]; P_br[k] = Pexp[order]

fig, ax = plt.subplots(figsize=(8.6, 5.8))
norm = plt.Normalize(-1.0, 1.0)
for i in range(2 * Md):                               # one multicoloured line per branch
    pts = np.array([lam_d, E_br[:, i]]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc = LineCollection(segs, cmap="coolwarm", norm=norm)
    lc.set_array(0.5 * (P_br[:-1, i] + P_br[1:, i]))  # segment colour = mean <P>
    lc.set_linewidth(1.5)
    ax.add_collection(lc)
cb = fig.colorbar(lc, ax=ax, pad=0.02)
cb.set_label(r"$\langle \hat P\rangle$ ")
cb.set_ticks([-1, 0, 1]); cb.set_ticklabels([r"$-1$ (odd)", r"$0$ (mixed)", r"$+1$ (even)"])
ax.set_xlim(lam_d[0], lam_d[-1]); ax.set_ylim(E_br.min() - 0.15, E_br.max() + 0.15)
ax.set_xlabel(r"$\lambda$"); ax.set_ylabel("single-charge eigenvalue")
ax.set_title(rf"Detuned dots $e_1={dots_d.e1},\ e_2={dots_d.e2}$ "
             rf"($\delta={dots_d.e1 - dots_d.e2:+.1f}$): broken parity"
             "\n" r"$\langle\hat P\rangle$ no longer $\pm1$; opposite-parity crossings open avoided gaps")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "rabi_spectrum_detuned_dots"+".png"), dpi=150, bbox_inches="tight"); plt.show()

### 6.1 — computing the parity expectation $\langle\hat P\rangle$

The colour scale above — and the parity colouring of the §6.1 spectra — is the expectation value
$\langle\psi|\hat P|\psi\rangle$ of the combined parity, evaluated for each eigenvector. Its closed form:

On the bare single-charge basis $|d,m\rangle$ (electron on dot $d\in\{1,2\}$, $m$ photons), the quadrature flip
$a\to-a$ is the photon parity $\hat\Pi=e^{i\pi a^\dagger a}$ with $\hat\Pi|m\rangle=(-1)^m|m\rangle$, and the dot
swap exchanges $|1,m\rangle\leftrightarrow|2,m\rangle$, so
$$\hat P\,|1,m\rangle=(-1)^m|2,m\rangle,\qquad \hat P\,|2,m\rangle=(-1)^m|1,m\rangle.$$
In the $(\mathrm{dot}1,\mathrm{dot}2)$ blocks $\hat P$ is therefore **purely off-diagonal** (no $|1,m\rangle\!\to\!|1,m\rangle$
part). Writing a single-charge eigenstate as
$$|\psi\rangle=\sum_m\big(L_m\,|1,m\rangle+R_m\,|2,m\rangle\big)$$
(the empty and double sectors carry no weight, charge being conserved), one has
$$\hat P|\psi\rangle=\sum_m(-1)^m\big(L_m|2,m\rangle+R_m|1,m\rangle\big),$$
and projecting back keeps only the surviving overlaps:
$$\langle\hat P\rangle=\langle\psi|\hat P|\psi\rangle=\sum_m(-1)^m\big(L_m^{*}R_m+R_m^{*}L_m\big)=2\,\mathrm{Re}\sum_m(-1)^m\,L_m^{*}R_m.$$
Only the **dot1–dot2 coherence** $L_m^{*}R_m$, weighted by the photon parity $(-1)^m$, contributes; the diagonal
weights $|L_m|^2,|R_m|^2$ cancel because $\hat P$ flips the dot. With $L_m,R_m$ the dot-1 / dot-2 amplitude blocks
of the eigenvector (`evec[M:2M]`, `evec[2M:3M]`):

```python
phot_par = (-1.0) ** np.arange(M)            # (-1)^m
Pexp = 2 * np.real(np.sum(phot_par[:, None] * np.conj(L) * R, axis=0))
```

Since $\hat P=\hat P^\dagger$ and $\hat P^2=\mathbb 1$, $\langle\hat P\rangle$ is **real and in $[-1,+1]$**: it is $\pm1$
exactly when $|\psi\rangle$ is a parity eigenstate (symmetric dots) and intermediate when detuning breaks the
symmetry. Checked against the brute-force $\psi^\dagger\hat P\psi$ with an explicitly built $\hat P$ to machine
precision ($\sim10^{-16}$): symmetric dots give $\langle\hat P\rangle=\pm1$ to all digits, while $e_1=0.6,\ e_2=0.7$
gives the continuous spread $\{-0.99,\dots,-0.55,\ +0.55,\dots,+0.99\}$ used in the plot above.

### 6.2. Deep strong limit

At $\lambda\gg1$ the single-charge spectra in both §6.1 plots collapse onto an **equally-spaced ladder**
of spacing $\omega$ and offset $\epsilon_0$, with the two parity ladders **merging pairwise**. This is the
*deep-strong-coupling* (DSC) limit, and it follows analytically from the Hamiltonian.

**1 — the hopping is a pseudospin in a precessing field.** Drop the constant onsite energy
$\epsilon_1=\epsilon_2\equiv\epsilon_0$ (symmetric dots) and treat the orbital (which-dot) degree of freedom as a
pseudospin. In the **dot basis** $\{|1\rangle,|2\rangle\}$,
$$\tau_x=|1\rangle\langle2|+|2\rangle\langle1|,\quad \tau_y=-i\big(|1\rangle\langle2|-|2\rangle\langle1|\big),\quad \tau_z=|1\rangle\langle1|-|2\rangle\langle2|,$$
while in the **bonding/antibonding** ($g/e$) basis $|g\rangle=\tfrac{1}{\sqrt2}(|1\rangle+|2\rangle),\ |e\rangle=\tfrac{1}{\sqrt2}(|1\rangle-|2\rangle)$,
$$\sigma_x=|g\rangle\langle e|+|e\rangle\langle g|,\quad \sigma_y=-i\big(|g\rangle\langle e|-|e\rangle\langle g|\big),\quad \sigma_z=|g\rangle\langle g|-|e\rangle\langle e|.$$
We work in the $g/e$ frame, where the bare hopping is diagonal, $t\,\tau_x=t\,\sigma_z$ (orbital splitting $\pm t$).

The two frames are two complementary readings of the same orbital qubit, related by the **Hadamard** change of
basis $|1\rangle,|2\rangle\leftrightarrow|g\rangle,|e\rangle$, under which
$$\boxed{\ \sigma_x=\tau_z,\qquad \sigma_y=-\tau_y,\qquad \sigma_z=\tau_x\ }$$
(the swap is derived in the aside). Each axis is a physical observable: $\tau_z$ is the **charge position /
dipole** (which dot), $\tau_x=\sigma_z$ is the **tunneling $=$ orbital energy** (the bonding–antibonding
splitting), and $\tau_y=-\sigma_y$ is the **inter-dot current**. So $\sigma_x=\tau_z$ and $\sigma_z=\tau_x$ are a
**population $\leftrightarrow$ coherence** duality — a *population* in one frame (charge on one dot, definite
$\tau_z$) is a *coherence* in the other (a fixed-phase $g$–$e$ superposition, off-diagonal $\sigma_x$), and vice
versa. The two bases are moreover **mutually unbiased**: each molecular state spreads with equal weight over both
dots ($|\langle1|g\rangle|^2=\tfrac12$), the orbital analogue of position vs. momentum.

This duality is exactly why the device behaves as a **charge qubit in a cavity**. In the lab frame (part 2)
the qubit gap is the tunneling $t\sigma_z=t\tau_x$, while the cavity couples through $\sigma_x=\tau_z$ — to the
**charge dipole**, the photon shaking the electron between dots (the standard electric-dipole light–matter
coupling). It also fixes the large-$\lambda$ physics: the tunneling favours an *energy* eigenstate (a delocalized
molecular orbital), the cavity dipole coupling favours a *position* eigenstate (localized charge $+$ displaced
photons), and since position and energy here are *maximally* complementary the two cannot be satisfied at once —
as $\lambda$ grows the coupling wins and the charge **self-traps** (part 3), the crossover being sharp precisely
because the bases are mutually unbiased.

Finally, the lone minus in $\sigma_y=-\tau_y$ is **forced, not incidental**: swapping two axes
($x\leftrightarrow z$) is a *reflection* ($\det=-1$), so the third axis must flip to keep the spin algebra
$[\sigma_x,\sigma_y]=2i\sigma_z$ intact. Its signature is physical — the flipped axis is the **current** $\tau_y$,
the one component **odd under time reversal**, so the mirror relating the localized and molecular descriptions
reverses the sense of circulation.

> **Where did the onsite energy go?** The dot term $\varepsilon_1 d_1^\dagger d_1+\varepsilon_2 d_2^\dagger d_2$ is
> *not* discarded. In the single-charge sector $d_1^\dagger d_1+d_2^\dagger d_2=\mathbb 1$, so for **symmetric dots**
> $\varepsilon_1=\varepsilon_2=\epsilon_0$ it collapses to the pure constant $\epsilon_0\,\mathbb 1$ — a rigid shift of every level.
> We pull it out here and **restore it as the $+\epsilon_0$ offset** of the ladder $E_n=\epsilon_0+n\omega$ (exactly why the
> resonant/off-resonance plots converge to $0.6+n$ and $1+n$). For **asymmetric dots** only the *average* drops,
> 
> $$\varepsilon_1 d_1^\dagger d_1+\varepsilon_2 d_2^\dagger d_2=\tfrac{\varepsilon_1+\varepsilon_2}{2}\,\mathbb 1+\tfrac{\varepsilon_1-\varepsilon_2}{2}\,\tau_z$$
> 
> and the detuning piece is genuine: since $\tau_z=\sigma_x$ in the $g/e$ frame it adds a
> $\tfrac{\varepsilon_1-\varepsilon_2}{2}\sigma_x$ term along the **same axis as the polaron coupling** in step 2,
> so it cannot be dropped. The symmetric case assumed here is what makes the whole onsite contribution a single offset.
>
> *Why $\tau_z=\sigma_x$?* The $\tau$'s are defined in the **dot basis** ($\tau_z=|1\rangle\langle1|-|2\rangle\langle2|$),
> the $\sigma$'s in the **$g/e$ basis** ($\sigma_x=|g\rangle\langle e|+|e\rangle\langle g|$). The localized dots
> $|1\rangle=(|g\rangle+|e\rangle)/\sqrt2,\ |2\rangle=(|g\rangle-|e\rangle)/\sqrt2$ are precisely the $\pm1$ eigenstates of
> $\sigma_x$, so the which-dot operator *is* $\sigma_x$: $\;|1\rangle\langle1|-|2\rangle\langle2|=|g\rangle\langle e|+|e\rangle\langle g|$.
> Equivalently the dot$\,\to g/e$ change of basis is a Hadamard, which swaps the Bloch axes
> $\tau_z\!\leftrightarrow\!\sigma_x$ and $\tau_x\!\leftrightarrow\!\sigma_z$ — the same rotation that turns the bare hopping
> $t\,\tau_x$ into the $g/e$ splitting $t\,\sigma_z$.

The single-charge block — **already in the polaron (Lang–Firsov) frame**, the electron–cavity dipole coupling having been absorbed into the displacement-dressed hopping — is
$$H=\omega a^\dagger a+t\big(\hat D|1\rangle\langle2|+\hat D^\dagger|2\rangle\langle1|\big)
=\omega a^\dagger a+t\cosh A\,\sigma_z-t\sinh A\,i\sigma_y,\qquad A=\lambda(a-a^\dagger).$$
Since $A$ is **anti-Hermitian**, write $A=i\Theta$ with $\Theta=\sqrt2\,\lambda\,p$ Hermitian
($p=(a-a^\dagger)/i\sqrt2$ the momentum quadrature); then $\cosh A=\cos\Theta,\ \sinh A=i\sin\Theta$, and
$$\boxed{\,H=\omega a^\dagger a+t\big(\cos\Theta\,\sigma_z+\sin\Theta\,\sigma_y\big),\quad \Theta=\sqrt2\,\lambda\,p\,}$$
— a pseudospin in a field of **fixed magnitude $t$** whose direction **precesses** in the $(z,y)$ plane
by the cavity-momentum-controlled angle $\Theta$.

**2 — reverse polaron transform $\to$ the lab-frame quantum Rabi model.** The block $H$ above is *already* the **polaron (Lang–Firsov) frame**; part 2 applies the **reverse** Lang–Firsov transform to undo the dressing and restore the **lab frame**, where the electron–cavity coupling reappears explicitly and **linearly** (the standard quantum Rabi model).

*Why transform.* In $H=\omega a^\dagger a+t(\cos\Theta\,\sigma_z+\sin\Theta\,\sigma_y)$ the pseudospin sees a
field of **fixed length $t$ but operator-valued direction** — the tilt $\Theta=\sqrt2\,\lambda\,p$ is a
*cavity* operator, so the spin's quantization axis depends on the photon state and the two cannot be
diagonalized separately. We move to a frame that **co-rotates with the field**, fixing the spin axis along
$z$ so the spin term collapses to the c-number $\pm t$. Because $\Theta$ is a cavity operator, this
"spin-following" rotation is a *spin-conditioned displacement* — the **(inverse) Lang–Firsov /
polaron transformation, run in reverse** to *leave* the polaron frame. The reward (part 3) is that the leftover cavity term becomes merely
**linear** in $a+a^\dagger$, i.e. a displacement, solved exactly by completing the square. Physically we boost
into the frame riding with the electron's photon cloud.

*Finding $R$ (geometry).* The field $t(\cos\Theta\,\hat z+\sin\Theta\,\hat y)$ has **constant length** and lies
in the $y$–$z$ plane, so it **precesses about $\hat x$** by the angle $\Theta$. The rotation about $x$ by
$\Theta$ undoes the tilt:
$$R=e^{+i\Theta\sigma_x/2}=\exp\!\Big[+\tfrac{\lambda}{2}(a-a^\dagger)\,\sigma_x\Big]\qquad\big(\text{using }\tfrac{i\Theta}{2}=\tfrac{\lambda}{2}(a-a^\dagger)\big),$$
and indeed $R^\dagger(\cos\Theta\,\sigma_z+\sin\Theta\,\sigma_y)R=\sigma_z$ — an $x$-rotation sends
$\sigma_z\!\to\!\cos\Theta\,\sigma_z-\sin\Theta\,\sigma_y$ and $\sigma_y\!\to\!\cos\Theta\,\sigma_y+\sin\Theta\,\sigma_z$,
and the two pieces recombine to $\sigma_z$. Since the rotation axis $\sigma_x=\tau_z$ is the **which-dot**
operator, $R$ displaces the cavity by $\mp\lambda/2$ according to the dot — the **orbital-conditional
displacement**. (Equivalently *Lang–Firsov*: pick $R$ to cancel the linear $(a+a^\dagger)\sigma_x$ coupling;
the same $R$ drops out.)

*Middle steps.* Work in the $\sigma_x$ eigenbasis $s=\pm1$ (the localized dots), where $R$ is a plain
displacement $R_s=D(-\tfrac{\lambda s}{2})$, hence $R^\dagger a R=a-\tfrac{\lambda}{2}\sigma_x$. The two terms
transform as (promote $s\to\sigma_x$ at the end, $s^2=1$)

$$\underbrace{R^\dagger(\omega a^\dagger a)R}_{\text{(A) cavity}}=\omega\Big(a^\dagger-\tfrac{\lambda}{2}\sigma_x\Big)\Big(a-\tfrac{\lambda}{2}\sigma_x\Big)=\omega a^\dagger a-\frac{\omega\lambda}{2}(a+a^\dagger)\sigma_x+\frac{\omega\lambda^2}{4},$$
$$\underbrace{R^\dagger H_{\mathrm{spin}}R}_{\text{(B) spin}}=t\,\sigma_z.$$

so summing and restoring $\epsilon_0$,
$$\tilde H=R^\dagger H R=\epsilon_0+\omega a^\dagger a+t\,\sigma_z-\frac{\omega\lambda}{2}(a+a^\dagger)\sigma_x+\frac{\omega\lambda^2}{4}.$$
**This $\tilde H$ is the lab-frame quantum Rabi model**: qubit splitting $2t$, explicit linear light–matter coupling $g=\omega\lambda/2$, with large
$\lambda$ its **deep-strong-coupling** regime $g\gg\omega,t$. It can be written in **either basis** — molecular $\sigma$ (qubit $t\sigma_z$, cavity coupling $\propto\sigma_x$) or site $\tau$ (qubit $t\tau_x$, cavity coupling $\propto\tau_z=$ the **charge dipole**), via $\sigma_z=\tau_x,\ \sigma_x=\tau_z$. (Reverse-polaron mapping $H\to\tilde H$ exact — verified to machine precision below.)

**3 — origin of the harmonicity (complete the square).** Why the **localized** basis and not the molecular
$g/e$ one? Completing the square needs the spin operator multiplying $(a+a^\dagger)$ to be *diagonal*, and that
operator is $\sigma_x$, whose eigenstates $s=\pm1$ are the localized dots — only then does the cavity see a
definite shift $-\tfrac{\lambda}{2}s$ to absorb. In this basis the **coupling is diagonal** while the qubit term
$t\sigma_z$ is purely **off-diagonal** (it flips dot $1\!\leftrightarrow\!2$), so $\tilde H$ is block-$2\times2$:

$$\tilde H=\epsilon_0+\omega a^\dagger a+t\,\tau_x-\frac{\omega\lambda}{2}(a+a^\dagger)\tau_z+\frac{\omega\lambda^2}{4}
= \epsilon_0+\omega (a^\dagger - \frac{\lambda}{2} \tau_z) (a - \frac{\lambda}{2} \tau_z)+ t \tau_x.$$
which rewrites as
$$\tilde H=\epsilon_0+\begin{pmatrix}\omega\,b_+^\dagger b_+ & t\\ t & \omega\,b_-^\dagger b_-\end{pmatrix},\qquad b_s=a-\tfrac{\lambda}{2}s,$$

where the off-diagonal $t$ (matrix elements $\propto e^{-\lambda^2/2}$, part 4) links the two ladders. Keeping the
**diagonal blocks** — i.e. setting $t\sigma_z$ aside for part 4 — and completing the square,
$$\tilde H_s=\omega a^\dagger a-\tfrac{\omega\lambda}{2}s(a+a^\dagger)+\tfrac{\omega\lambda^2}{4}=\omega\,b_s^\dagger b_s,$$
the displacement energy $-\omega\lambda^2/4$ **exactly cancels** the added $+\omega\lambda^2/4$, so each block is a
*displaced* oscillator with the **same, $\lambda$-independent** spectrum $\omega n$. Restoring $\epsilon_0$,
$$E_n\ \longrightarrow\ \epsilon_0+\omega n,\qquad n=0,1,2,\dots\quad(\text{two-fold degenerate}).$$
**This is the harmonicity:** the only $\lambda$-dependent term is a conditional *displacement*, which merely shifts
the oscillator origin and leaves the ladder $\{\omega n\}$ untouched; the neglected $t\sigma_z$ is the
exponentially small inter-block coupling, reinstated next. **Note — this is the *lab* frame (the Rabi model $\tilde H$):** the
displaced oscillators $b_s$ are the genuine lab-frame deep-strong states (displaced "cats"). The spectrum $\epsilon_0+\omega n$ is frame-independent, but the
*eigenstates* here are displaced; part 5 maps to the **polaron frame $H$** — the one the notebook diagonalizes — where the same states read as **bare** Fock.

Which basis is "right" is fixed by **which term dominates**, i.e. by the regime:

| basis | diagonal term | natural regime | zeroth-order states (lab frame) |
|---|---|---|---|
| molecular $g/e$ ($\sigma_z$) | qubit splitting $t\sigma_z$ | small $\lambda$ ($t\gg g$) | bonding/antibonding **polaritons** (§6.1, JC) |
| localized dot ($\sigma_x$) | coupling $g(a+a^\dagger)\sigma_x$ | large $\lambda$ ($g=\tfrac{\omega\lambda}{2}\gg t$) | **displaced oscillators** $b_\pm$ |

Deep strong coupling ($g\gg t$): the coupling wins, so in the lab frame the $\sigma_x$-diagonal displaced
oscillators are the natural zeroth-order states and $t\sigma_z$ is a small perturbation that $\hat P$-symmetrises
them into quasi-degenerate doublets (part 4). What they are *physically* — bare Fock states in the polaron frame — is
part 5.

**4 — what splits the doublets (and the residual anharmonicity).** The qubit term $+t\sigma_z$ is
**off-diagonal** in the dot basis: it flips $s$, coupling the oppositely-displaced ladders $b_\pm$ with
Franck–Condon amplitude $\langle n|\hat D(\lambda)|n'\rangle\sim e^{-\lambda^2/2}$. Hence
- each degenerate doublet splits by $\approx 2t\,e^{-\lambda^2/2}$ (the residual bonding/antibonding);
- second-order inter-rung repulsion leaves a tiny anharmonicity.

Both vanish $\propto e^{-\lambda^2/2}$. Because the rung-$n$ factor is $e^{-\lambda^2/2}L_n(\lambda^2)$ and the
Laguerre $L_n$ **grows with $n$**, higher rungs decouple only once $\lambda^2\gtrsim O(n)$ — so the ladder
**harmonizes rung-by-rung from the bottom up**, approaching $\omega$ from below.

**5 — back to the polaron frame: the states are bare Fock.** Transform the lab-frame Rabi result back to the polaron frame,
$H=R\,\tilde H\,R^\dagger$, with $R=e^{+\frac{\lambda}{2}(a-a^\dagger)\sigma_x}$. Two middle steps do all the work:
$$R\,a\,R^\dagger=a+\tfrac{\lambda}{2}\sigma_x,$$
$$R\,\sigma_z\,R^\dagger=\hat D(\lambda)\,|1\rangle\langle2|+\hat D^\dagger(\lambda)\,|2\rangle\langle1|.$$
The first **un-displaces the cavity**: the diagonal harmonic blocks become the *bare* ladder,
$R\,(\omega b_s^\dagger b_s)\,R^\dagger=\omega a^\dagger a$ (the $\pm\omega\lambda^2/4$ constants cancel). The second
**re-grows the displacement out of $\sigma_z$**: the constant off-diagonal $t$ turns back into the full
Franck–Condon hopping $t\hat D(\lambda)$. Together they rebuild the original
$H=\epsilon_0+\omega a^\dagger a+t(\hat D(\lambda)|1\rangle\langle2|+\text{h.c.})$.

So in the **polaron frame $H$** — the frame the notebook diagonalizes — the harmonic ladder $\epsilon_0+\omega n$ is the **bare cavity Fock ladder** (photons
**undisplaced**, $\langle n\rangle=n$), and the large-$\lambda$ eigenstates are the **bare
molecular–Fock products** $|g,n\rangle,|e,n\rangle$. The displaced "cats" are how the *same* states appear in the lab frame $\tilde H$ (the familiar deep-strong Rabi ground state) — the Lang–Firsov displacement is exactly what undoes the photon cloud. The tunneling is **Franck–Condon
quenched**, $t_{\mathrm{eff},n}=t\,\langle n|\hat D(\lambda)|n\rangle=t\,e^{-\lambda^2/2}L_n(\lambda^2)\to0$, so the
bonding–antibonding splitting collapses and each pair $\{|g,n\rangle,|e,n\rangle\}$ becomes degenerate at
$\epsilon_0+\omega n$, split only by $2t_{\mathrm{eff},n}$ — the part-4 factor, now read as the polaron-frame
$g$–$e$ splitting. Numerically at $\lambda=5$ every state is $>99.8\%$ a single bare product:

| rung $n$ | $P=-1$ | $P=+1$ |
|---|---|---|
| 0 | $\lvert\tilde g,0\rangle=\lvert g,0\rangle$ | $\lvert1^-\rangle=\lvert e,0\rangle$ |
| 1 | $\lvert2^-\rangle=\lvert e,1\rangle$ | $\lvert1^+\rangle=\lvert g,1\rangle$ |
| 2 | $\lvert2^+\rangle=\lvert g,2\rangle$ | $\lvert3^-\rangle=\lvert e,2\rangle$ |
| 3 | $\lvert4^-\rangle=\lvert e,3\rangle$ | $\lvert3^+\rangle=\lvert g,3\rangle$ |

— exactly the coalescing doublets $\{|g,n\rangle,|e,n\rangle\}$ read off the §6.1 spectra
($\{|\tilde g,0\rangle,|1^-\rangle\}$, $\{|1^+\rangle,|2^-\rangle\}$, and in general $\{|n^+\rangle,|(n{+}1)^-\rangle\}$).
The states **re-purify** onto single bare products as $\lambda$ grows (ground weight on $|g,0\rangle$:
$0.96\to0.994\to0.9995$ at $\lambda=1,3,5$), opposite to the maximal mixing near $\lambda\sim1$.

(The §6.2 decomposition cell below prints the weights, $\langle n\rangle$, and the FC-split check.) The residual
$g$–$e$ splitting organises these bare-Fock states into **double-well tunnel doublets** — part 6.

**6 — the double-well doublet.** *Where the picture comes from.* The two dots are, literally, a **symmetric
double-well potential** for the charge, and the hidden parity $\hat P=(1\!\leftrightarrow\!2)\times(a\to-a)$ is its
**left–right reflection**. Every symmetric double well organises its spectrum into **tunnel doublets** — a lower
symmetric and a higher antisymmetric state split by the tunnelling amplitude — and that is exactly the
$\{|g,n\rangle,|e,n\rangle\}$ structure we found. What is special here is that the inter-well tunnelling is
**Franck–Condon (polaron) renormalised** by the cavity.

*The effective Hamiltonian.* Project onto one rung — the degenerate pair $\{|1,n\rangle,|2,n\rangle\}$ (charge on
dot 1 / dot 2 with $n$ **bare** photons, the two wells, exchanged by $\hat P$). These two are *exactly* degenerate because $H_0=\epsilon_0+\omega a^\dagger a$ is dot-blind; a partner
$|2,m\rangle$ with $m\neq n$ is off-resonant by $(n-m)\omega$, so the photon-number-changing part of the dressed
hopping connects to it only at **second order** — the small part-4 anharmonicity, never the doublet splitting.
Within the degenerate pair the dressed hopping of part 5 reduces to a two-level double-well Hamiltonian
$$H_n=(\epsilon_0+\omega n)\,\mathbb 1-t_{\mathrm{eff},n}\,\tau_x,\qquad t_{\mathrm{eff},n}=t\,\langle n|\hat D(\lambda)|n\rangle=t\,e^{-\lambda^2/2}L_n(\lambda^2),$$
whose eigenstates are the symmetric/antisymmetric combinations $|g,n\rangle=\tfrac1{\sqrt2}(|1,n\rangle+|2,n\rangle)$,
$|e,n\rangle=\tfrac1{\sqrt2}(|1,n\rangle-|2,n\rangle)$, split by $2|t_{\mathrm{eff},n}|$. So $\tau_x$ (inter-dot
tunnelling) is the well-to-well coupling and the harmonic index $n$ merely labels which doublet.

*Why these states, even for vanishing tunnelling.* The two diagonal entries are equal (the degeneracy), so the
two-level mixing angle **saturates**, $\tan2\theta=2t_{\mathrm{eff},n}/(E_1-E_2)\to\infty\Rightarrow\theta=\tfrac\pi4$:
the eigenstates are **maximally** delocalised $50/50$ superpositions *however small* $t_{\mathrm{eff},n}$ is —
quenching the tunnelling shrinks the **gap** $2|t_{\mathrm{eff},n}|$ to zero, not the mixing (contrast the
non-degenerate case, where a small coupling barely rotates the states). Equivalently $\hat P$ **swaps the wells**,
$\hat P|1,n\rangle=(-1)^n|2,n\rangle$, so the exact eigenstates must be parity-definite — symmetric or antisymmetric —
and the localised $|1,n\rangle,|2,n\rangle$ are not $\hat P$-eigenstates, hence not stationary. It is ordinary
bonding/antibonding formation: degeneracy fixes the *states* (maximal mixing, by symmetry), the coupling sets only
the *splitting*.

*What it entails.* **(i) Exponential splitting = a tunnelling barrier.** $2t_{\mathrm{eff}}\sim 2t\,e^{-\lambda^2/2}$:
the cavity coupling acts as a barrier of WKB action / **Huang–Rhys factor** $S=\lambda^2/2$ — the squared
separation $\lambda$ of the two dots' photon clouds (the $\pm\lambda/2$ displacements of part 2). This is precisely
the **Holstein / molecular-polaron Franck–Condon band-narrowing**: the cavity is the vibrational mode, $\lambda$ the
electron–phonon coupling, and $e^{-\lambda^2/2}$ the Franck–Condon (Debye–Waller) factor that quenches tunnelling.
**(ii) Dynamical self-trapping.** A charge prepared in one well $|1,n\rangle=\tfrac1{\sqrt2}(|g,n\rangle+|e,n\rangle)$
oscillates to the other with period $T_n=\pi/|t_{\mathrm{eff},n}|\sim(\pi/t)\,e^{+\lambda^2/2}$ — exponentially
long. That, *not* a localised eigenstate, is the precise meaning of "self-trapping": the eigenstates stay the
delocalised parity states $|g,n\rangle,|e,n\rangle$; only the *dynamics* freezes. **(iii) A photon-tunable barrier
and coherent destruction of tunnelling.** Because the barrier carries the Laguerre factor $L_n(\lambda^2)$, it is
**photon-number dependent** and **changes sign at the zeros of $L_n$**. At those special $\lambda$ the splitting
vanishes — the doublet closes and the parity ordering of $|g,n\rangle/|e,n\rangle$ **flips** — a photon-resolved
**coherent destruction of tunnelling** (the cusps in the part-3 splitting figure are exactly these nodes).

In short, the coalescing pairs you read off the §6.1 spectra are the tunnel doublets of a **photon-dressed
symmetric double well**: a real-space double well whose tunnelling is Franck–Condon narrowed by the cavity, giving
an exponentially small, photon-number-modulated splitting on each harmonic rung.

The cell below (a) verifies the reverse-polaron mapping $H\to\tilde H$ to the lab-frame Rabi model is exact and (b) checks the large-$\lambda$ spectrum
against the harmonic ladder $\epsilon_0+n\omega$ — including the rung-by-rung convergence and the cutoff-independence.

In [ ]:
# 6.2 — deep-strong limit: (a) verify the polaron -> quantum-Rabi mapping is exact;
#       (b) check the large-lambda spectrum is the harmonic ladder  E = e + n*omega.

def _sc_levels(lam, e, t, N, w=1.0):
    """Sorted single-charge eigenvalues (the 2(N+1) levels in columns N+1..3N+2)."""
    ev, _, _ = HamiltonianBuilder(CavityParameters(n=N, omega=w, kappa=1e-3),
                                  DotParameters(e1=e, e2=e, t=t), lam).diagonalize()
    return np.sort(ev[N + 1:3 * (N + 1)].real)

def _rabi_polaron(lam, e, t, N, w=1.0):
    """Lab-frame quantum Rabi model (reverse polaron transform of H)  H~ = e + w a†a + t sz - (w*lam/2)(a+a†) sx + w*lam^2/4."""
    M = N + 1
    a = np.diag(np.sqrt(np.arange(1, M)), 1); ad = a.T; n = ad @ a; I = np.eye(M)
    sz = np.array([[1., 0], [0, -1]]); sx = np.array([[0., 1], [1, 0]]); I2 = np.eye(2)
    H = ((e + w * lam**2 / 4) * np.kron(I, I2) + w * np.kron(n, I2)
         + t * np.kron(I, sz) - (w * lam / 2) * np.kron(a + ad, sx))
    return np.sort(np.linalg.eigvalsh(H))

# (a) exact mapping, lowest 18 levels, Fock space large enough that truncation is irrelevant
print("polaron -> Rabi mapping  (max |Δ| over lowest 18 single-charge levels):")
for lam in (0.5, 1.0, 2.0, 3.0):
    A = _sc_levels(lam, 0.6, 0.5, N=120); B = _rabi_polaron(lam, 0.6, 0.5, N=120)
    print(f"   lambda={lam:>3}:  {np.max(np.abs(A[:18] - B[:18])):.1e}")

# (b) harmonic ladder: pair levels into doublets -> centers, spacings, intra-doublet splitting
    
    '''
    1. E = _sc_levels(...) — gets the sorted single-charge eigenvalues from the physical
      (polaron-frame) dressed-hopping Hamiltonian.
    2. cen (doublet centers) — pairs adjacent levels (E[0],E[1]), (E[2],E[3]), … and averages
      each pair. These are the rung centers, predicted to fall at ε₀ + nω. It keeps the lowest
      ndoub=6 doublets.
    3. spl (intra-doublet splitting) — the gap within each pair, E[2m+1] − E[2m]. This is the
      2·t_eff,n = 2t⟨n|D(λ)|n⟩ = 2t e^{−λ²/2}L_n(λ²) Franck–Condon splitting that goes
      exponentially to zero (part 4).
    4. np.diff(cen) (center spacings) — the rung-to-rung spacing, predicted to approach ω from
      below as λ grows (part 3).
    '''
    
def harmonics(lam, e, t, N=60, ndoub=6, w=1.0):
    E = _sc_levels(lam, e, t, N, w)
    cen = np.array([(E[2*m] + E[2*m+1]) / 2 for m in range(ndoub)])
    spl = np.array([E[2*m+1] - E[2*m]       for m in range(ndoub)])
    return cen, spl, np.diff(cen)

print("\nharmonic ladder at lambda=5  (predicted: center = e + n,  spacing = omega = 1):")
for tag, e, t in (("resonant   (e=0.6, t=0.5)", 0.6, 0.5), ("off-reson. (e=1.0, t=1.0)", 1.0, 1.0)):
    cen, spl, sp = harmonics(5.0, e, t)
    print(f"  {tag}")
    print("     (center-e)/omega : " + " ".join(f"{x-e:6.3f}" for x in cen))
    print("     center spacings  : " + " ".join(f"{x:6.4f}" for x in sp))
    print(f"     rung-0 split = {spl[0]:.2e}   vs  2t e^(-lam^2/2) = {2*t*np.exp(-25/2):.2e}")

print("\ntruncation (resonant, lambda=5) — low-rung spacings are converged already at the plot's n=8:")
for N in (8, 20, 60):
    _, _, sp = harmonics(5.0, 0.6, 0.5, N=N)
    print(f"   n_max={N:>2}: " + " ".join(f"{x:6.4f}" for x in sp))

# figure: (L) spacings -> omega rung by rung;  (C) the exponentially small doublet splitting (log);
#         (R) the same splitting on a linear axis.
lam_grid = np.linspace(0.0, 5.0, 60)
spac_by_rung = [[] for _ in range(5)]; split_by_rung = [[] for _ in range(5)]
for lam in lam_grid:
    _, spl, sp = harmonics(lam, 0.6, 0.5, N=40, ndoub=6)
    for m in range(5):
        spac_by_rung[m].append(sp[m]); split_by_rung[m].append(spl[m])

fig, (axL, axR, axLin) = plt.subplots(1, 3, figsize=(16, 4.2))
for m in range(5):
    axL.plot(lam_grid, spac_by_rung[m], lw=1.3, label=rf"$E_{{{m+1}}}\!-\!E_{{{m}}}$")
    axR.semilogy(lam_grid, np.abs(split_by_rung[m]), lw=1.3, label=f"rung {m}")
    axLin.plot(lam_grid, np.abs(split_by_rung[m]), lw=1.3, label=f"rung {m}")
axL.axhline(1.0, color="k", ls=":", lw=1.2, label=r"$\omega$")
axL.set_xlabel(r"$\lambda$"); axL.set_ylabel("doublet-center spacing")
axL.set_title(r"Approach to harmonicity: level spacings $=\omega$")
axL.set_ylim(0.7, 1.55); axL.legend(fontsize=8, ncol=2, loc="upper right"); axL.grid(alpha=0.3)
axR.semilogy(lam_grid, 2 * 0.5 * np.exp(-lam_grid**2 / 2), "k--", lw=1.6, label=r"$2t_{\mathrm{eff},0}=2t\,e^{-\lambda^2/2}$")
axR.set_xlabel(r"$\lambda$"); axR.set_ylabel(r"doublet splitting $2|t_{\mathrm{eff},n}|$  (log axis)")
axR.set_title(r"Doublet splitting: exponentially small in $\lambda^2$")
axR.set_ylim(1e-7, 3); axR.legend(fontsize=8, ncol=2, loc="lower left"); axR.grid(alpha=0.3, which="both")
axLin.plot(lam_grid, 2 * 0.5 * np.exp(-lam_grid**2 / 2), "k--", lw=1.6, label=r"$2t\,e^{-\lambda^2/2}$")
axLin.set_xlabel(r"$\lambda$"); axLin.set_ylabel(r"doublet splitting $2|t_{\mathrm{eff},n}|$  (linear axis)")
axLin.set_title(r"Doublet splitting: linear axis")
axLin.legend(fontsize=8, ncol=2, loc="upper right"); axLin.grid(alpha=0.3)
plt.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, "deep_strong_harmonicity.png"), dpi=150, bbox_inches="tight"); plt.show()


**Reading the verification cell.** It backs the §6.2 claims numerically and draws the three diagnostic panels.

- **`_sc_levels(λ,e,t,N)`** — diagonalises the **polaron-frame** dressed-hopping model (`HamiltonianBuilder`) — the frame the whole notebook works in
  and returns the sorted single-charge eigenvalues (`ev[N+1:3(N+1)]` = the `2(N+1)` one-electron levels). These
  are the §6.1 spectra — the ground truth.
- **`_rabi_polaron(λ,e,t,N)`** — builds the **lab-frame** quantum Rabi model ($\tilde H=R^\dagger H R$, the reverse polaron transform of $H$)
  $\tilde H=\epsilon_0+\omega a^\dagger a+t\sigma_z-\tfrac{\omega\lambda}{2}(a+a^\dagger)\sigma_x+\tfrac{\omega\lambda^2}{4}$
  *from scratch* (just $a,a^\dagger,\sigma$ — no $\hat D$). Comparing its spectrum to `_sc_levels` tests that the
  reverse-polaron mapping $H\to\tilde H$ (part 2) is exact.
- **`harmonics(λ,e,t,N)`** — pairs consecutive levels into rung doublets and returns the doublet **centres**
  ($\to\epsilon_0+\omega n$), the intra-doublet **splittings** ($\to 2t_{\mathrm{eff},n}$), and the
  centre-to-centre **spacings** ($\to\omega$) — i.e. the inter-rung gap vs the intra-rung gap.

**Why doublet *centres*, not raw consecutive spacings.** In the deep-strong limit the spectrum is a stack of
*near-degenerate pairs*, $E_{2m}\approx\epsilon_0+m\omega-t_{\mathrm{eff},m}$ and
$E_{2m+1}\approx\epsilon_0+m\omega+t_{\mathrm{eff},m}$. Raw consecutive differences therefore **alternate** between
two unlike quantities — the **intra**-doublet gap $E_{2m+1}-E_{2m}=2t_{\mathrm{eff},m}\to0$ and the **inter**-doublet
gap $E_{2m+2}-E_{2m+1}=\omega-t_{\mathrm{eff},m+1}-t_{\mathrm{eff},m}\to\omega$ — a $0,\omega,0,\omega,\dots$ zigzag
that never settles on a single value even though the system *is* harmonising. The "spacing $=\omega$" claim is a
statement about the **rung centres**: averaging each pair, $(E_{2m}+E_{2m+1})/2$, cancels the $\pm t_{\mathrm{eff},m}$
and leaves the clean ladder $\epsilon_0+m\omega$ whose differences approach $\omega$ monotonically. That is why
harmonicity is tested with **two independent quantities** — centres $\to\epsilon_0+m\omega$ *and* splittings $\to0$ —
rather than one raw-spacing plot, which would entangle the two. (The pairing $(E_{2m},E_{2m+1})$ is the genuine
doublet only once $\lambda$ is large enough to group the levels that way; at intermediate $\lambda$ the two parity
ladders interleave in energy — the §6.1 caveat that energy-ordered $|n\pm\rangle$ are not the Rabi partners — so this
construction is meaningful precisely in the deep-strong regime. The schematic below makes the pairing explicit.)

**Printed checks.** (a) $\max|\,$`_sc_levels`$-$`_rabi_polaron`$|\approx10^{-13}$ (mapping exact, part 2);
(b) at $\lambda=5$ the centres sit at $\epsilon_0+n$ and spacings at $\omega$ for resonant *and* off-resonance
(part 3), and the rung-0 splitting matches $2t\,e^{-\lambda^2/2}$ (part 4); (c) $n_{\max}=8,20,60$ give the same
low-rung spacings (not a cutoff artefact).

**Figure (swept over $\lambda$).** *Left* — the inter-rung **centre spacings** $E_{m+1}-E_m$ converging to
$\omega$ **from below**, higher rungs lagging (rung-by-rung harmonisation). *Centre* — the intra-doublet
**splitting** $2|t_{\mathrm{eff},n}|=|E_{2m+1}-E_{2m}|$ on a **log $y$-axis** (the $x=\lambda$ axis stays linear;
`semilogy` log-scales only the ordinate, and the plotted data is the splitting itself, *not* $\log t_{\mathrm{eff}}$),
plunging along the dashed $2t_{\mathrm{eff},0}=2t\,e^{-\lambda^2/2}$; the higher-rung **cusps** are the Laguerre
zeros $L_n(\lambda^2)=0$ (coherent destruction of tunnelling, part 6). *Right* — the **same splitting on a linear
axis**, where the Gaussian collapse to $\approx0$ by $\lambda\gtrsim2$ is obvious but the per-rung structure and
cusps are washed out — the two splitting panels are complementary (linear shows the overall collapse, log reveals
the fine structure).

**Numerical confirmation (deep-strong limit).**

- **Offset & spacing.** At $\lambda=5$ the doublet centers sit at $\epsilon_0+n$ to $\lesssim10^{-2}$ and the center
  spacings equal $\omega$ to $\sim0.1\%$ on the low rungs — the predicted $\epsilon_0+n\omega$ ladder, with each rung
  a two-fold (opposite-parity) doublet.
- **Detuning-independent.** Resonant ($\epsilon_0{=}0.6$) and off-resonance ($\epsilon_0{=}1$) both collapse to $\epsilon_0+n\omega$;
  only the offset $\epsilon_0$ differs. Since the effective hopping $t_{\rm eff}\!\to\!0$, the ladder forgets the
  detuning $\Delta\omega=\omega-2t$ entirely.
- **Doublet (parity) splitting $=2t\,e^{-\lambda^2/2}$** to four decimals ($3.9\times10^{-6}$ measured vs
  $3.7\times10^{-6}$ predicted at $\lambda=5$) — the residual tunneling between the two localized wells
  (right panel hugs the dashed line at large $\lambda$).
- **Rung-by-rung.** The spacings approach $\omega$ **from below**, low rungs first (left panel); higher rungs
  lag because their Franck–Condon factor $e^{-\lambda^2/2}L_n(\lambda^2)$ is larger ($L_n$ grows with $n$),
  so they need a bigger $\lambda^2$ to decouple. This is **not** a truncation effect: $n_{\max}=8$ and $60$
  give the same low-rung spacings, so the converged low levels in the §6.1 plots are genuine physics.

**Schematic of the doublet ladder (illustration).** The figure below is a *cartoon*, not a diagonalisation: it
draws levels $E_0\dots E_7$ vs $\lambda$ from the deep-strong form $E_{2n},E_{2n+1}=\epsilon_0+n\omega\mp
t_{\mathrm{eff},n}$, with the splitting **enlarged** so it stays visible. Each rung $n$ is a doublet whose **centre**
(faint dashed line) is pinned at $\epsilon_0+n\omega$ — so the centre-to-centre spacing is $\omega$ — while the
**intra-doublet splitting** $2t_{\mathrm{eff},n}=2t\,e^{-\lambda^2/2}$ (vertical arrow) collapses as $\lambda$ grows.
This is exactly the structure the `harmonics` pairing exploits: *average* each pair to read off the harmonic ladder,
*difference* each pair to read off the vanishing splitting.

In [ ]:
# 6.2 — SCHEMATIC of the doublet ladder (ILLUSTRATION ONLY — not a diagonalization).
# Levels E_0..E_7 vs λ: each rung n is a doublet centred at ε₀+nω (faint dashed line),
# split by 2 t_eff,n = 2t e^{-λ²/2} (t drawn enlarged so the splitting stays visible).
# Shows how the pairs form and how the intra-doublet splitting shrinks while the centres
# stay locked at ε₀+nω.

lam = np.linspace(0.0, 3.0, 300)
e0, w, t = 0.0, 1.0, 0.30                  # schematic numbers (t enlarged for visibility)
half = t * np.exp(-lam**2 / 2)             # half the intra-doublet splitting, ∝ e^{-λ²/2}

fig, ax = plt.subplots(figsize=(7.8, 6.0))
ndoub = 4                                  # rungs n=0..3  ->  levels E_0 .. E_7
for n in range(ndoub):
    c = e0 + n * w
    lo, hi = c - half, c + half
    ax.plot(lam, np.full_like(lam, c), color="0.7", lw=0.9, ls="--", zorder=1)   # doublet centre
    ax.plot(lam, lo, color="C0", lw=2.0, zorder=2)                               # E_{2n}
    ax.plot(lam, hi, color="C3", lw=2.0, zorder=2)                               # E_{2n+1}
    ax.text(-0.10, lo[0], rf"$E_{{{2*n}}}$",   va="center", ha="right", color="C0", fontsize=10)
    ax.text(-0.10, hi[0], rf"$E_{{{2*n+1}}}$", va="center", ha="right", color="C3", fontsize=10)
    ax.text(3.06, c, rf"$\epsilon_0+{n}\,\omega$", va="center", ha="left", color="0.5", fontsize=9)

# annotate ONE intra-doublet splitting (rung n=1) with a double-headed arrow
xm, n_mark = 0.55, 1
cm, hm = e0 + n_mark * w, t * np.exp(-xm**2 / 2)
ax.annotate("", xy=(xm, cm + hm), xytext=(xm, cm - hm),
            arrowprops=dict(arrowstyle="<->", color="k", lw=1.4))
ax.text(xm + 0.07, cm, r"$2t_{\mathrm{eff},n}=2t\,e^{-\lambda^2/2}$", va="center", ha="left", fontsize=9)

# annotate the centre-to-centre spacing = ω (at large λ where the doublets have merged)
xs = 2.6
ax.annotate("", xy=(xs, e0 + 2 * w), xytext=(xs, e0 + 1 * w),
            arrowprops=dict(arrowstyle="<->", color="0.4", lw=1.4))
ax.text(xs - 0.07, e0 + 1.5 * w, r"$\omega$", va="center", ha="right", color="0.4", fontsize=11)

ax.set_xlim(-0.35, 3.6); ax.set_ylim(-0.6, 3.6)
ax.set_xlabel(r"$\lambda$"); ax.set_ylabel("energy (schematic)")
ax.set_title("Schematic: single-charge levels collapse into harmonic doublets "
             r"$\{|g,n\rangle,|e,n\rangle\}$" "\n"
             r"(illustration — splitting enlarged; centres fixed at $\epsilon_0+n\omega$)")
ax.set_yticks([e0 + n for n in range(ndoub)]); ax.grid(alpha=0.2)
plt.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, "deep_strong_doublet_schematic.png"), dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# 6.2 — large-λ structure in the LAB frame.  The physical eigenstates become essentially BARE
# molecular–Fock products |g,n>,|e,n> (photons undisplaced, <n>=n), and each {|g,n>,|e,n>} doublet
# splits by the Franck–Condon factor  2t<n|D(λ)|n> = 2t e^{-λ²/2} L_n(λ²).
from numpy.polynomial.laguerre import lagval
def _Ln(n, x):
    return lagval(x, [0]*n + [1])

def large_lambda_structure(lam=5.0, e=0.6, t=0.5, N=90, nshow=8):
    ev, evec, _ = HamiltonianBuilder(CavityParameters(n=N, omega=1.0, kappa=1e-3),
                                     DotParameters(e1=e, e2=e, t=t), lam).diagonalize()
    Ms = N + 1
    tags = label_rabi_partners(ev, evec, Ms)
    order = sorted(tags, key=lambda k: ev[k].real)[:nshow]
    p0 = evec[:, order[0]]
    use_minus = abs((p0[Ms:2*Ms] - p0[2*Ms:3*Ms])[0]) >= abs((p0[Ms:2*Ms] + p0[2*Ms:3*Ms])[0])
    print(f"polaron-frame eigenstates at lambda={lam}  (e={e}, t={t}):  bare-product weight and mean photon <n>")
    print(f"  {'state':>7} {'E':>8} {'P':>3} {'<n>':>6}   dominant bare product")
    for k in order:
        psi = evec[:, k]; L, R = psi[Ms:2*Ms], psi[2*Ms:3*Ms]
        g  = (L - R) / np.sqrt(2) if use_minus else (L + R) / np.sqrt(2)
        eo = (L + R) / np.sqrt(2) if use_minus else (L - R) / np.sqrt(2)
        wg, we = np.abs(g)**2, np.abs(eo)**2
        nbar = float(sum(m * (abs(L[m])**2 + abs(R[m])**2) for m in range(Ms)))
        m = int(np.argmax(np.maximum(wg, we)))
        orb, w = ('g', wg[m]) if wg[m] >= we[m] else ('e', we[m])
        print(f"  {tags[k]['tag']:>7} {ev[k].real:8.4f} {int(tags[k]['parity']):>+3} {nbar:6.3f}   {w:.3f} |{orb},{m}>")
    E = np.sort(ev[Ms:3*Ms].real)
    print("\n  {|g,n>,|e,n>} doublet splitting   vs   2t e^(-lam^2/2)|L_n(lam^2)| = 2t|<n|D(lam)|n>|:")
    for n in range(4):
        meas = E[2*n+1] - E[2*n]; fc = 2*t*np.exp(-lam**2/2)*abs(_Ln(n, lam**2))
        print(f"    rung n={n}:   split = {meas:.3e}    FC = {fc:.3e}")

large_lambda_structure(lam=5.0)


## 7. Deep strong transport

Transport companion to §6.2: what does the I–V curve look like once $\lambda\gg1$, where the
inter-dot hopping is Franck–Condon quenched, $t_{\mathrm{eff}}=t\,e^{-\lambda^2/2}\to0$?

**No new physics classes are needed.** We reuse the §4–§5 machinery unchanged:
`HamiltonianBuilder` (§3) $\to$ `CavityDecayMatrix` / `TunnelRateMatrix` / `RateEquationSolver`
(§4) $\to$ `TransportCalculator` (§5). The drivers below — `deep_strong_iv` and its
independent-lead generalisation `deep_strong_iv_fc` — are exactly the sequential-tunneling
pipeline of §10's `iv_curve_sweep`, written out inline because that helper is defined later. The
only numerical change for $\lambda\gg1$ is the **Fock cutoff**: $\hat D$ spreads weight over
$\sim\lambda^2$ photon states, so we take $N=3\,\max(\lambda,\lambda_L,\lambda_R)^2+10$ per curve
(the §8 top-rung truncation caveat).

The figure is a **2×2 grid** ($\kappa=1$, resonant $e=0.6,\ t=0.5,\ \omega=1$) that separates the
role of the **inter-dot** dressing $\lambda$ (in $\hat D(\lambda)$ on the hopping) from the **lead**
dressings $\lambda_L,\lambda_R$ (in $\hat D(\lambda_\ell)\,d_\ell^\dagger$ on injection / extraction):

- **Bare leads** (top-left) — $\lambda_L=\lambda_R=0$, sweep $\lambda\in\{1,2,3\}$: pure inter-dot dressing.
- **Dressed leads** (top-right) — $\lambda_L=\lambda_R=\lambda$, sweep $\lambda$, **plus** the fixed-lead
  control $\lambda_L=\lambda_R=1,\ \lambda\in\{2,3\}$ (dashed) that holds the lead dressing while the
  inter-dot one grows.
- **Left lead only** (bottom-left) — $\lambda=\lambda_R=0$, sweep $\lambda_L$: dress only the *injection* lead.
- **Right lead only** (bottom-right) — $\lambda=\lambda_L=0$, sweep $\lambda_R$: dress only the *extraction* lead.

See the observations below for what each panel isolates: sum-rule protection, the lead-vs-inter-dot
distinction, and the injection-vs-extraction directionality.

In [ ]:
# 7 — Deep strong transport: I-V at lambda >> 1.  Four scenarios isolating the roles of the
# inter-dot dressing (lam) and the lead dressings (lam_L, lam_R).  deep_strong_iv IS the
# sequential-tunneling pipeline of iv_curve_sweep (sec 10), inlined; deep_strong_iv_fc is the
# same with INDEPENDENT lead displacements.  Only the sec 3-5 classes are used.

def deep_strong_iv(dots, cavity, leads_template, lam, V_values, *, dress_leads):
    """One-sided-bias (mu_L=V, mu_R=0) Rabi I(V).  dress_leads=True sets lam_L=lam_R=lam."""
    eigvals, eigvecs, _ = HamiltonianBuilder(cavity, dots, lam).diagonalize()
    Gamma_ph = CavityDecayMatrix(cavity, eigvals, eigvecs).build()
    transp = TransportCalculator(cavity, leads_template)
    I = np.empty(len(V_values))
    for j, V in enumerate(V_values):
        leads = copy.deepcopy(leads_template)
        leads.mu_L, leads.mu_R = V, 0.0
        if dress_leads:
            leads.lam_L = leads.lam_R = lam
        G_L, _G_R, G_tot = TunnelRateMatrix(cavity, dots, leads, eigvals, eigvecs).build()
        P = RateEquationSolver(G_tot, Gamma_ph, cavity).steady_solver()
        I[j] = transp.compute_current(G_L, P)
    return I


def deep_strong_iv_fc(dots, cavity, leads_template, lam, V_values, *, lam_L, lam_R):
    """As deep_strong_iv but with INDEPENDENT lead displacements (lam = inter-dot dressing)."""
    eigvals, eigvecs, _ = HamiltonianBuilder(cavity, dots, lam).diagonalize()
    Gamma_ph = CavityDecayMatrix(cavity, eigvals, eigvecs).build()
    transp = TransportCalculator(cavity, leads_template)
    I = np.empty(len(V_values))
    for j, V in enumerate(V_values):
        leads = copy.deepcopy(leads_template)
        leads.mu_L, leads.mu_R = V, 0.0
        leads.lam_L, leads.lam_R = lam_L, lam_R
        G_L, _G_R, G_tot = TunnelRateMatrix(cavity, dots, leads, eigvals, eigvecs).build()
        P = RateEquationSolver(G_tot, Gamma_ph, cavity).steady_solver()
        I[j] = transp.compute_current(G_L, P)
    return I


def _Nds(lam):
    return max(8, int(np.ceil(3 * lam**2)) + 10)        # Fock cutoff ~ 3*lam^2 + buffer

# --- parameters: resonant bright-bonding dot (eps_g = e - t = 0.1 at lambda=0), kappa = 1 ---
omega_ds = 1.0
dots_ds  = DotParameters(e1=0.6, e2=0.6, t=0.5)
leads_ds = LeadParameters(gamma_L=1e-3, gamma_R=1e-3, mu_L=0.0, mu_R=0.0, mu_0=0.0,
                          T_L=1e-3, T_R=1e-3, lam_L=0.0, lam_R=0.0)
lam_ds   = [1.0, 2.0, 3.0]
V_ds     = np.linspace(-0.3, 5.0, 240)
kappa_ds = 1.0

def _cav(*lams):
    return CavityParameters(n=_Nds(max(lams)), omega=omega_ds, kappa=kappa_ds)   # cutoff from the largest displacement

# lambda = 0 non-interacting reference (lam = lam_L = lam_R = 0)
I_ref = deep_strong_iv(dots_ds, CavityParameters(n=8, omega=omega_ds, kappa=kappa_ds),
                       leads_ds, 0.0, V_ds, dress_leads=False)

cmap = plt.cm.viridis(np.linspace(0.15, 0.82, len(lam_ds)))

# --- panel data ---
bare      = {lam: deep_strong_iv(dots_ds, _cav(lam), leads_ds, lam, V_ds, dress_leads=False)
             for lam in lam_ds}
dressed   = {lam: deep_strong_iv(dots_ds, _cav(lam), leads_ds, lam, V_ds, dress_leads=True)
             for lam in lam_ds}
# scenario 3: leads fixed (lam_L=lam_R=1), inter-dot lam = 2, 3  (merged into dressed panel)
scen3     = {lam: deep_strong_iv_fc(dots_ds, _cav(lam, 1.0), leads_ds, lam, V_ds, lam_L=1.0, lam_R=1.0)
             for lam in (2.0, 3.0)}
# scenario 1: only LEFT lead dressed (lam = lam_R = 0), sweep lam_L
leftonly  = {k: deep_strong_iv_fc(dots_ds, _cav(k), leads_ds, 0.0, V_ds, lam_L=k, lam_R=0.0)
             for k in (1.0, 2.0, 3.0)}
# scenario 2: only RIGHT lead dressed (lam = lam_L = 0), sweep lam_R
rightonly = {k: deep_strong_iv_fc(dots_ds, _cav(k), leads_ds, 0.0, V_ds, lam_L=0.0, lam_R=k)
             for k in (1.0, 2.0, 3.0)}

# --- figure: 2x2 ---
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, sharey=True)
(axB, axD), (axL, axR) = axes
def _ref(ax): ax.plot(V_ds, I_ref, color="0.6", ls="--", lw=1.2, label=r"$\lambda=0$ (non-int.)")

_ref(axB)
for lam, col in zip(lam_ds, cmap):
    axB.plot(V_ds, bare[lam], color=col, lw=1.7, label=rf"$\lambda={lam:g}$")
axB.set_title(r"Bare leads  ($\lambda_L=\lambda_R=0$, sweep $\lambda$)")

_ref(axD)
for lam, col in zip(lam_ds, cmap):
    axD.plot(V_ds, dressed[lam], color=col, lw=1.7, label=rf"$\lambda_L=\lambda_R=\lambda={lam:g}$")
for lam, col in zip((2.0, 3.0), ("tab:brown", "tab:pink")):
    axD.plot(V_ds, scen3[lam], color=col, lw=1.7, ls="--",
             label=rf"$\lambda_L=\lambda_R=1,\ \lambda={lam:g}$")
axD.set_title(r"Dressed leads ($\lambda_L=\lambda_R=\lambda$) + fixed leads ($\lambda_L=\lambda_R=1$)")

_ref(axL)
for k, col in zip((1.0, 2.0, 3.0), cmap):
    axL.plot(V_ds, leftonly[k], color=col, lw=1.7, label=rf"$\lambda_L={k:g}$")
axL.set_title(r"Left lead only  ($\lambda=\lambda_R=0$, sweep $\lambda_L$)")

_ref(axR)
for k, col in zip((1.0, 2.0, 3.0), cmap):
    axR.plot(V_ds, rightonly[k], color=col, lw=1.7, label=rf"$\lambda_R={k:g}$")
axR.set_title(r"Right lead only  ($\lambda=\lambda_L=0$, sweep $\lambda_R$)")

for ax in axes.flat:
    ax.grid(alpha=0.3); ax.legend(fontsize=7.5, loc="center right")
for ax in (axL, axR): ax.set_xlabel(r"$V_b$")
for ax in (axB, axL): ax.set_ylabel(r"current $I$")
fig.suptitle(r"Deep-strong transport: I–V at $\kappa=1$, resonant ($e=0.6,\ t=0.5,\ \omega=1$); "
             r"roles of inter-dot $\lambda$ vs lead $\lambda_L,\lambda_R$", y=1.005)
fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, "deep_strong_iv_grid.png"), dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# 7 — Fock-cutoff convergence: the in-window I-V steps must not move with N.
# Recompute the most demanding case (lambda=3) at two cutoffs; bare and dressed.
V_chk = np.linspace(-0.3, 5.0, 130)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for N_chk, ls in ((_Nds(3.0), "-"), (_Nds(3.0) + 16, "--")):
    cav = CavityParameters(n=N_chk, omega=omega_ds, kappa=kappa_ds)
    for dress, col in ((False, "tab:blue"), (True, "tab:red")):
        I = deep_strong_iv(dots_ds, cav, leads_ds, 3.0, V_chk, dress_leads=dress)
        ax.plot(V_chk, I, ls=ls, color=col, lw=1.5,
                label=f"N={N_chk}, {'dressed' if dress else 'bare'}")
ax.set_xlabel(r"$V_b$"); ax.set_ylabel(r"$I$"); ax.grid(alpha=0.3)
ax.set_title(rf"Cutoff convergence at $\lambda=3$ "
             rf"($N={_Nds(3.0)}$ solid vs $N={_Nds(3.0)+16}$ dashed)")
ax.legend(fontsize=8, loc="center right")
fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, "deep_strong_iv_convergence.png"), dpi=150, bbox_inches="tight"); plt.show()

### 7 — observations: lead vs inter-dot dressing in the deep-strong limit

- **Bare leads — no exponential blockade (sum rule).** At $\lambda\in\{1,2,3\}$ the conducting
  current stays $O(1)$ (first plateau essentially $\lambda$-independent). The bare injection vertex
  $d_\ell^\dagger$ is photon-conserving, so the empty $\to$ single-charge weight obeys
  $\sum_\alpha|\langle\alpha|d_\ell^\dagger|0,0\rangle|^2=1$; the inter-dot dressing only redistributes
  that unit weight among the (now nearly degenerate) $0$-photon polaritons, which remain delocalised
  across both dots and therefore still conduct. The FC quench of $t_{\mathrm{eff}}$ shrinks the
  bonding/antibonding **gap**, not the conductance.
- **Step merger + threshold saturation.** As $\lambda$ grows the LP/UP steps (split by $2t_{\mathrm{eff}}$)
  coalesce into a single step, and the $\varepsilon_g=\epsilon_0-t_{\mathrm{eff}}$ onset drifts up toward $\epsilon_0$ —
  the bare-lead I–V is the transport image of the §6.2 doublet collapse.
- **The onset channel is the dressed polaron $|\tilde g,0\rangle$, not the bare $|g,0\rangle$.** The first
  step is injection from the empty dot $|0,0\rangle$ ($E=0$) into the *lowest single-charge eigenstate*, so
  it turns on at $V_{\mathrm{onset}}=E(|\tilde g,0\rangle)$. The bare ket $|g,0\rangle$ (bonding $\otimes$
  vacuum) has a *fixed* energy $\epsilon_0-t=0.1$ and is **not** an eigenstate at $\lambda>0$; the eigenstate is the
  polaron $|\tilde g,0\rangle\approx|g,0\rangle+$ (few-% photon cloud), and $\varepsilon_g$ denotes *its*
  energy — the renormalised $\epsilon_0-t_{\mathrm{eff}}$, not the bare $\epsilon_0-t$. **Why the onset climbs with $\lambda$:**
  the hopping is FC-dressed, $t_{\mathrm{eff}}=t\langle0|\hat D(\lambda)|0\rangle=t\,e^{-\lambda^2/2}$, so the
  bonding ground sits a distance $t_{\mathrm{eff}}$ *below* the bare onsite $\epsilon_0$; as $\lambda$ grows the FC
  factor quenches $t_{\mathrm{eff}}\to0$, the bonding stabilisation vanishes, and the level rises from $\epsilon_0-t$
  up to $\epsilon_0$ (Holstein/Franck–Condon band narrowing — the molecule loses its binding energy). Verified onsets
  $E(|\tilde g,0\rangle)$:

  | $\lambda$ | 0 | 0.5 | 1 | 2 | 3 |
  |---|---|---|---|---|---|
  | **onset** $E(|\tilde g,0\rangle)$ | 0.100 | 0.131 | 0.217 | 0.452 | 0.562 |
  | bare $\epsilon_0-t$ | 0.100 | 0.100 | 0.100 | 0.100 | 0.100 |
  | FC est. $\epsilon_0-t\,e^{-\lambda^2/2}$ | 0.100 | 0.159 | 0.297 | 0.532 | 0.594 |

  climbing $0.10\to0.56$ toward $e=0.6$. The first-order estimate $e-t\,e^{-\lambda^2/2}$ **overshoots** (0.297
  vs 0.217 at $\lambda=1$) because $|\tilde g,0\rangle$ also repels downward off $|e,1\rangle$ at second order;
  the marker uses the exact eigenvalue. (At $\lambda=1$ the LP/UP $|1^\mp\rangle$ steps above the onset are
  still resolved since $2t_{\mathrm{eff}}\approx0.4$; they merge by $\lambda=2$–$3$ as $t_{\mathrm{eff}}\to0$.)
- **Dressed leads — Franck–Condon blockade.** Switching on $\lambda_L=\lambda_R=\lambda$ multiplies the
  elastic line by $e^{-\lambda^2}$ ($\sim10^{-1},10^{-2},10^{-4}$ at $\lambda=1,2,3$), so the dressed panel
  collapses toward zero. The missing spectral weight moves into $\omega$-spaced photon sidebands at higher
  bias, but under one-sided bias both leads dressed leaves the *extraction* leg Pauli-frozen (its sidebands
  need the electron below $\mu_R=0$), so the recovery saturates low rather than returning to the bare value
  — the §12 directionality, taken to large $\lambda$.
- **Decomposing the dressing — lead vs inter-dot, injection vs extraction (2×2 figure).** High-bias
  plateau currents:

  | $\lambda$ or $\lambda_\ell$ | 1 | 2 | 3 |
  |---|---|---|---|
  | **bare** ($\lambda_L{=}\lambda_R{=}0$, sweep $\lambda$) | 0.96 | 0.98 | 0.995 |
  | **dressed** ($\lambda_L{=}\lambda_R{=}\lambda$) | 0.52 | 0.03 | $\sim$0 |
  | **left only** ($\lambda_L$, inject) | 0.97 | 0.50 | 0.03 |
  | **right only** ($\lambda_R$, extract) | 0.69 | 0.10 | $\sim$0 |
  | **fixed leads** ($\lambda_L{=}\lambda_R{=}1$, sweep $\lambda$) | 0.52 | 0.49 | 0.52 |

  Three readings:
  - **The blockade is a *lead* effect, not the inter-dot dressing.** The fixed-lead row stays flat at
    $\approx0.5$ while the inter-dot $\lambda$ runs $1\to3$: cranking $\hat D(\lambda)$ on the *hopping* only
    reshuffles the parity-protected, still-delocalised eigenstates and barely moves the current, whereas
    $\hat D(\lambda_\ell)$ on a *lead* multiplies the elastic line by $e^{-\lambda_\ell^2}$. Same sum-rule logic
    as the bare panel (and as §12's $(1,0,0)$ control).
  - **Injection vs extraction directionality.** Dressing only the *left* (injection) lead **recovers** with
    bias — the missing elastic weight returns as an $\omega$-spaced sideband **staircase** ($\lambda_L{=}1\to0.97$),
    because the injected electron can shed photons into the empty single-charge manifold as $V$ opens. Dressing
    only the *right* (extraction) lead instead **saturates flat** after the first step ($\lambda_R{=}1\to0.69$):
    its sidebands would need the outgoing electron *below* $\mu_R=0$, which is Pauli-blocked, so there is no
    recovery. This is the §12 directionality, here isolated at $\lambda=0$ so it is purely a lead effect.
  - **Magnitudes.** At equal dressing the left (inject) plateau beats the right (extract) at every $\lambda_\ell$
    ($0.97/0.50/0.03$ vs $0.69/0.10/{\sim}0$); dressing *both* (the dressed row) is the most suppressed,
    inheriting the frozen-extraction ceiling.
- **Convergence.** The in-window steps are unchanged between $N=3\lambda^2+10$ and $N+16$ (convergence
  cell): the features are physics, not truncation. Trusting higher photon sidebands needs a few extra Fock
  states of buffer, exactly the §8 top-rung rule.

### 7 — note: persistent current and the secular limit

The bare-lead onset climbs to $e$ and the LP/UP steps merge as $t_{\mathrm{eff}}=t\,e^{-\lambda^2/2}\to0$:
the molecular doublet $\{|g,n\rangle,|e,n\rangle\}$ collapses onto the bare onsite $e$, doubly degenerate.
Yet the plateau current stays $\approx1$ for *every* $\lambda$ tested — even though the cavity has all but
switched off the inter-dot hopping. Two things resolve the apparent paradox.

**Degenerate $\neq$ isolated.** The cavity quenches the *gap*, not the *delocalisation*. The eigenstates are
parity-protected sym/antisym combinations $|g,n\rangle=(|1,n\rangle+|2,n\rangle)/\sqrt2$,
$|e,n\rangle=(|1,n\rangle-|2,n\rangle)/\sqrt2$ with **maximal $45^\circ$ mixing independent of $t_{\mathrm{eff}}$**
(equal diagonals $\Rightarrow \tan2\theta=2t_{\mathrm{eff}}/0\to\infty$). So an electron injected on dot 1 lands
in a state that still lives on dot 2, and both lead overlaps stay
$\langle g,0|d_1^\dagger|0,0\rangle=\langle 0,0|d_2|g,0\rangle=1/\sqrt2$ for all $\lambda$. The relevant
comparison is $2t_{\mathrm{eff}}$ vs $\gamma$, **not** $t_{\mathrm{eff}}$ vs $0$: while $2t_{\mathrm{eff}}\gg\gamma$
the dots are *not* effectively isolated — the (slow) molecular bridge still empties faster than the lead, since
the hop time $\tau_{\mathrm{hop}}\sim e^{+\lambda^2/2}/t$ stays below the dwell time $1/\gamma$. This is the
dynamical-only self-trapping of §6.2 part 6.

**The secular artifact.** `RateEquationSolver` propagates only eigenstate *populations*. In the eigenbasis the
channels stay delocalised, so it keeps conducting at $I\sim\gamma$-limited (full height) for arbitrarily large
$\lambda$ — even past $\lambda_c$ where $2t_{\mathrm{eff}}=\gamma$ (here $\lambda_c\approx3.7$; see figure). The
gap drops three orders of magnitude below $\gamma$ by $\lambda=5$ while the plateau barely moves. Physically,
once the doublet is degenerate to within $\gamma$ the coherences $|g,0\rangle\langle e,0|$ can no longer be
dropped, and the standard serial double-dot result $I\approx t_{\mathrm{eff}}^2/\gamma\propto e^{-\lambda^2}$
takes over — the localised-basis statement *isolated dots $\Rightarrow$ no current*. Capturing that crossover
needs a **non-secular** (Redfield/Lindblad keeping the degenerate coherences) treatment, or working in the
localised $|1,n\rangle,|2,n\rangle$ basis. **The persistent full current at $\lambda=4,5$ below is therefore
partly a secular artifact**, trustworthy only for $\lambda\lesssim3$ ($2t_{\mathrm{eff}}\gg\gamma$).

In [ ]:
# 7 — diagnostic: bare first-plateau current pushed into the deep-strong regime (lambda up to 5).
# The parity-protected molecular doublet stays delocalised, so the SECULAR rate equation keeps the
# current lead-limited (~1) even after the gap 2 t_eff drops below gamma at lambda_c (where 2 t_eff = gamma).
# A non-secular treatment would instead give I ~ t_eff^2/gamma there. Single plateau point per lambda
# (V well above the e-onset), reusing the sec 7 parameters (dots_ds, leads_ds, kappa_ds, _Nds).

def _bare_plateau(lam, V=2.0):
    cav = CavityParameters(n=_Nds(lam), omega=omega_ds, kappa=kappa_ds)
    leads = copy.deepcopy(leads_ds); leads.mu_L, leads.mu_R = V, 0.0
    ev, evec, _ = HamiltonianBuilder(cav, dots_ds, lam).diagonalize()
    Gph = CavityDecayMatrix(cav, ev, evec).build()
    G_L, _, G_tot = TunnelRateMatrix(cav, dots_ds, leads, ev, evec).build()
    P = RateEquationSolver(G_tot, Gph, cav).steady_solver()
    return TransportCalculator(cav, leads).compute_current(G_L, P)

lam_ext = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
I_ext   = np.array([_bare_plateau(l) for l in lam_ext])
t_ds, gamma = dots_ds.t, leads_ds.gamma_L
gap_ratio = 2 * t_ds * np.exp(-lam_ext**2 / 2) / gamma      # 2 t_eff / gamma
lam_c = np.sqrt(2 * np.log(2 * t_ds / gamma))               # 2 t_eff = gamma

fig, axL = plt.subplots(figsize=(8, 4.8))
axL.plot(lam_ext, I_ext, "o-", color="tab:blue", lw=1.9, ms=6,
         label=r"bare plateau $I$ (secular RE)")
axL.axhline(1.0, color="0.75", ls=":", lw=1.0)
axL.set_xlabel(r"$\lambda$"); axL.set_ylabel(r"plateau current $I$", color="tab:blue")
axL.tick_params(axis="y", labelcolor="tab:blue"); axL.set_ylim(0, 1.12)
axR = axL.twinx()
axR.semilogy(lam_ext, gap_ratio, "s--", color="tab:red", lw=1.6, ms=5,
             label=r"$2t_{\mathrm{eff}}/\gamma$")
axR.axhline(1.0, color="tab:red", ls=":", lw=1.0)
axR.set_ylabel(r"$2t_{\mathrm{eff}}/\gamma$  (log)", color="tab:red")
axR.tick_params(axis="y", labelcolor="tab:red")
axL.axvline(lam_c, color="k", ls="-.", lw=1.3)
axL.text(lam_c + 0.06, 0.30,
         rf"$2t_{{\mathrm{{eff}}}}=\gamma$" "\n" rf"$\lambda_c\approx{lam_c:.2f}$"
         "\n(secular RE breaks\n down beyond here)", fontsize=8.5, va="top")
axL.set_title(r"Persistent current vs the gap-to-$\gamma$ crossover (bare leads, $\kappa=1$)")
h1, l1 = axL.get_legend_handles_labels(); h2, l2 = axR.get_legend_handles_labels()
axL.legend(h1 + h2, l1 + l2, loc="center left", fontsize=8.5)
fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, "deep_strong_persistent_current.png"), dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# 7 — bare-leads I-V deep into the strong-coupling regime (lambda = 3, 4, 5).
# Full I-V companion to the persistent-current diagnostic: the curves collapse onto a
# SINGLE step at the bare onsite e = 0.6 (LP/UP merged, eps_g -> e) and still reach full
# current -- the molecular doublet stays delocalised (secular RE; see the note above).
# Reuses deep_strong_iv and the sec 7 parameters; cutoff N = 3 lambda^2 + 10 per curve.
V_hi   = np.linspace(-0.3, 2.2, 130)
lam_hi = [3.0, 4.0, 5.0]
I_ref_hi = deep_strong_iv(dots_ds, CavityParameters(n=8, omega=omega_ds, kappa=kappa_ds),
                          leads_ds, 0.0, V_hi, dress_leads=False)
cmap_hi = plt.cm.plasma(np.linspace(0.15, 0.72, len(lam_hi)))

fig, ax = plt.subplots(figsize=(7.6, 4.8))
ax.plot(V_hi, I_ref_hi, color="0.6", ls="--", lw=1.3, label=r"$\lambda=0$ (non-int.: $e\mp t$)")
for lam, col in zip(lam_hi, cmap_hi):
    cav = CavityParameters(n=_Nds(lam), omega=omega_ds, kappa=kappa_ds)
    I = deep_strong_iv(dots_ds, cav, leads_ds, lam, V_hi, dress_leads=False)
    ax.plot(V_hi, I, color=col, lw=1.9, label=rf"$\lambda={lam:g}\ (N={_Nds(lam)})$")
ax.axvline(0.6, color="k", ls=":", lw=1.1); ax.text(0.62, 0.08, r"$e=0.6$", fontsize=9)
ax.set_xlabel(r"$V_b$"); ax.set_ylabel(r"current $I$"); ax.grid(alpha=0.3)
ax.set_ylim(-0.03, 1.12)
ax.set_title(r"Bare-leads I–V, deep strong ($\lambda=3,4,5$): single step at the bare onsite $e$")
ax.legend(fontsize=8.5, loc="center right")
fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, "deep_strong_bare_hi_lambda.png"), dpi=150, bbox_inches="tight"); plt.show()

## 8. Rabi vs Jaynes–Cummings spectrum comparison

Diagonalise the strong-coupling **Rabi** Hamiltonian (`HamiltonianBuilder`, this notebook) and the weak-coupling **JC** Hamiltonian (`JCBuilder`, copied from `JupyterDots/weak_coupling.ipynb`) over the same λ-range, then plot the single-particle sector (indices `M..3M`) on a shared axis.

At small λ the two spectra coincide; at λ ≳ ω/2 the counter-rotating terms missing in JC produce visible deviations (Bloch–Siegert shift).


In [ ]:
# JCBuilder copied verbatim from weak_coupling.ipynb (Cell 0) so this
# cell stays self-contained and doesn't need `import_ipynb`.

class JCBuilder:
    def __init__(self, cavity: CavityParameters, dots: DotParameters, lam):
        self.cavity = cavity     # store the whole cavity object
        self.dots = dots
        self.lam = lam
        
        # phonon cutoff
        self.n = self.cavity.n
        self.M = self.n + 1
        self.dim = 4 * self.M
        
        # Build the full Hamiltonian once and store it
        self.H = self.build()
        
        self.H0 = self._build_sector_0()
        self.H1 = self._build_sector_1()
        self.H2 = self._build_sector_2()
    
    """Now let's build the Hamiltonian step-by-step."""
    # -----------------------------
    # Sector builders
    # -----------------------------

    def _build_sector_0(self):
        """Constructs the zero-particle sector Hamiltonian."""
        H0 = np.zeros((self.M, self.M), dtype=np.float64)
        for n in range(self.M):
            H0[n, n] = self.cavity.omega * n
        return H0

    def _build_sector_2(self):
        """Constructs the two-particle sector Hamiltonian."""
        H2 = np.zeros((self.M, self.M), dtype=np.float64)
        for n in range(self.M):
            H2[n, n] = self.dots.e1 + self.dots.e2 + self.cavity.omega * n
        return H2

    def _build_sector_1(self):
        """Constructs the single-particle sector Hamiltonian in the basis:
          n=photon number  ===> basis = {|g,0>; |e,0>,|g,1>; |e,1>,|g,2>; ...; |e,n-1>, |g,n>; ... |e,n>}
        """
        eg = (self.dots.e1 + self.dots.e2)/2 - np.sqrt((self.dots.e1 - self.dots.e2)**2/4 + (self.dots.t)**2)
        ee = (self.dots.e1 + self.dots.e2)/2 + np.sqrt((self.dots.e1 - self.dots.e2)**2/4 + (self.dots.t)**2)
        
        H1 = np.zeros((2 * self.M, 2 * self.M), dtype=np.float64)

        # diagonal  block
        for n in range(self.M):
            H1[2*n, 2*n] = eg + self.cavity.omega * n
            H1[2*n+1, 2*n+1] = ee + self.cavity.omega * n

        # Hopping
        for m in range(self.n):
            r = 2*m + 1
            c = r + 1
            H1[r, c] = np.sqrt(m+1)*self.lam
            H1[c, r] = np.sqrt(m+1)*self.lam

        return (H1 + H1.conj().T) / 2.0

    # -----------------------------
    # Full Hamiltonian
    # -----------------------------

    def build(self):
        H0 = self._build_sector_0()
        H1 = self._build_sector_1()
        H2 = self._build_sector_2()

        H = np.zeros((self.dim, self.dim), dtype=np.float64)

        idx0 = list(range(self.M))
        idx1 = list(range(self.M, 3*self.M))
        idx2 = list(range(3*self.M, 4*self.M))

        # Fill blocks
        H[np.ix_(idx0, idx0)] = H0
        H[np.ix_(idx1, idx1)] = H1
        H[np.ix_(idx2, idx2)] = H2

        return (H + H.conj().T) / 2.0

    # -----------------------------
    # Diagonalization
    # -----------------------------

    def diagonalize(self):
    
        M = self.M
        dim = self.dim
    
        # --- eigenvalues of diagonal sectors ---
        eigvals_0 = np.diag(self.H0)
        eigvals_2 = np.diag(self.H2)
    
        # eigenvectors are identity
        eigvecs_0 = np.eye(M)
        eigvecs_2 = np.eye(M)
    
        # --- diagonalize JC sector ---
        eigvals_1, eigvecs_1 = eigh(self.H1)
    
        # --- embed eigenvectors into full Hilbert space ---
        V0 = np.zeros((dim, M), dtype=np.complex128)
        V1 = np.zeros((dim, 2*M), dtype=np.complex128)
        V2 = np.zeros((dim, M), dtype=np.complex128)
    
        V0[0:M, :] = eigvecs_0
        V1[M:3*M, :] = eigvecs_1
        V2[3*M:4*M, :] = eigvecs_2
    
        # --- merge spectra ---
        eigvals = np.concatenate([eigvals_0, eigvals_1, eigvals_2])
        eigvecs = np.hstack([V0, V1, V2])
    
        # --- charge sector labels ---
        state_charge = (
            [0]*M +
            [1]*(2*M) +
            [2]*M
        )
    
        return eigvals, eigvecs, state_charge

    def diagonalize_ordered(self):

        M = self.M
        dim = self.dim
        
        # --- eigenvalues of diagonal sectors ---
        eigvals_0 = np.diag(self.H0)
        eigvals_2 = np.diag(self.H2)
        
        eigvecs_0 = np.eye(M)
        eigvecs_2 = np.eye(M)
        
        # --- diagonalize JC sector ---
        eigvals_1, eigvecs_1 = eigh(self.H1)
        
        # --- reorder eigenstates to match original basis ---
        overlaps = np.abs(eigvecs_1)**2
        dominant_idx = np.argmax(overlaps, axis=0)
        order = np.argsort(dominant_idx)
        
        eigvals_1 = eigvals_1[order]
        eigvecs_1 = eigvecs_1[:, order]
        
        # --- embed eigenvectors ---
        V0 = np.zeros((dim, M), dtype=np.complex128)
        V1 = np.zeros((dim, 2*M), dtype=np.complex128)
        V2 = np.zeros((dim, M), dtype=np.complex128)
        
        V0[0:M, :] = eigvecs_0
        V1[M:3*M, :] = eigvecs_1
        V2[3*M:4*M, :] = eigvecs_2
        
        eigvals_ordered = np.concatenate([eigvals_0, eigvals_1, eigvals_2])
        eigvecs_ordered = np.hstack([V0, V1, V2])
        
        state_charge_ordered = (
            [0]*M +
            [1]*(2*M) +
            [2]*M
        )
        
        return eigvals_ordered, eigvecs_ordered, state_charge_ordered
            
#################
##### PRIME 
#################

# ----------------------------------------------------------
# Sweep lambda, collect sector-1 spectrum from both models
# ----------------------------------------------------------
dots = DotParameters(e1=0.6, e2=0.6, t=0.5)
cavity = CavityParameters(n=5, omega=1.0, kappa=1e-3)

M = cavity.n + 1              # photon-block size
lam_values = np.linspace(0.0, 0.5, 200)

spectra_Rabi = np.empty((len(lam_values), 2 * M))
spectra_JC = np.empty((len(lam_values), 2 * M))

for k, lam in enumerate(lam_values):
    # Rabi (full counter-rotating terms)
    rabi = HamiltonianBuilder(cavity, dots, lam)
    evals_R, *_ = rabi.diagonalize()
    spectra_Rabi[k] = evals_R[M:3 * M]

    # Jaynes-Cummings (rotating-wave approximation).
    # The polaron expansion gives g_JC = t * lam_Rabi, so feed the JC
    # builder dots.t * lam to share a common physical coupling axis.
    
    jc = JCBuilder(cavity, dots, dots.t * lam)
    evals_J, *_ = jc.diagonalize_ordered()
    spectra_JC[k] = evals_J[M:3 * M]

# -------------------------------------------------------------------
# Plot: Rabi as dashed lines, JC as markers; matched colors per level
# -------------------------------------------------------------------
n_levels = spectra_Rabi.shape[1]
colors = plt.cm.viridis(np.linspace(0.0, 0.95, n_levels))

fig, ax = plt.subplots(figsize=(8, 5))
for i in range(n_levels):
    ax.plot(lam_values, spectra_Rabi[:, i],
            linestyle="-", lw=1.0, color=colors[i], alpha=0.9)
    ax.plot(lam_values, spectra_JC[:, i],
            linestyle="-.", marker="o", markersize=2.5,
            markevery=8, color=colors[i], alpha=0.9)

# Legend handles: one entry per model, color-agnostic
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], linestyle="-", color="black",
           label="Rabi (strong coupling)"),
    Line2D([0], [0], linestyle="-.", marker="o", markersize=4,
           color="black", label="JC (RWA)"),
]

ax.legend(handles=legend_handles, loc="upper left", frameon=True)
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel(r"Eigenvalue (sector 1)")
ax.set_title(rf"Rabi vs JC spectrum (JC at $g=t\lambda$)  "
             rf"($e_1=e_2={dots.e1}$, $t={dots.t}$, "
             rf"$\omega={cavity.omega}$, $N={M-1}$)")

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "rabi_vs_jc_spectrum"+".png"), dpi=150, bbox_inches="tight"); plt.show()


### 8 — observation: the top rung is a photon-cutoff artifact

With cutoff `n_cut=5` (12 single-charge levels) the Rabi and JC ladders match
well for λ < 0.5 — **except the topmost rung, which departs already at λ ≳ 0.2**.
This is a Fock-truncation boundary effect, not real physics.

- **Rabi:** the full displacement $e^{\lambda(a-a^\dagger)}$ couples each photon
  level to its neighbours with strength $\propto \lambda\sqrt{n+1}$. The highest
  in-window level ($n=n_\mathrm{cut}$) *should* hybridise with the $n_\mathrm{cut}+1$
  level above it, but that level is truncated away — so its eigenvalue is missing
  the level-repulsion from above and is **unconverged**. The error grows as
  $\lambda\sqrt{n_\mathrm{cut}}$, hence it shows up early (top-level
  $|{\rm Rabi}-{\rm JC}|\approx0.08,\,0.17,\,0.29$ at $\lambda=0.2,\,0.3,\,0.4$,
  versus $\sim0.01$ for the bulk rungs).
- **JC (RWA):** the top state $|e,n_\mathrm{cut}\rangle$ has its only RWA partner
  $|g,n_\mathrm{cut}+1\rangle$ truncated away, so JC leaves it as an *exact,
  uncoupled bare level* at $\varepsilon_e + n_\mathrm{cut}\,\omega$ (a flat,
  λ-independent line). JC's top is "clean", Rabi's is truncation-contaminated, so
  they diverge precisely at the top.
- **Bulk rungs** have both partners well inside the window and agree to the
  genuine $O(\lambda^2)$ Bloch–Siegert difference ($\sim0.01$ at $\lambda=0.3$).

**It is the cutoff boundary, not λ:** raising the cutoff to $n_\mathrm{cut}=12$,
the level that was the deviating top at $n_\mathrm{cut}=5$ ($E\approx6.1$, diff
$0.17$) becomes interior and its diff collapses to $\sim0.02$; the large deviation
simply moves to the new top rung. **Rule of thumb:** the top ~1–2 rungs at any
cutoff are truncation artifacts — to trust the spectrum up to photon level $N$,
keep a few Fock states of buffer above it.


## 9. Rabi vs Jaynes–Cummings I–V comparison

Compare the I–V curves of the full polaron **Rabi** model with the weak-coupling
**JC** model (`jc_wc.py`, extracted from `wk_coupl_refactored.ipynb`) at three
resonance conditions. Two **fixed-κ** grids — §9a at κ = 0 and §9b at κ = 1 —
each sweeping λ (rows) across the three detunings (columns).

**Coupling convention.** The physical coupling is `g`. In the Rabi model
`g = λ` (its input), while in the JC model `g = t·λ`; so the JC builder is fed
`t·λ` whenever the Rabi model is fed `λ`. The two should coincide at small λ
(benchmark) and diverge for λ ≳ 1 (RWA breakdown).

**Setup.** Bonding ground fixed at ε_g = 0.1 (bright bonding) via
`e1 = e2 = 0.1 + t`; three `t` give Δω = ω − 2t = {+0.2, 0, −0.2}. Leads
undressed (λ_L = λ_R = 0), one-sided bias (μ_L = V, μ_R = 0). κ = 0 uses a
1e-12 proxy (strict 0 gives a degenerate steady state once photons populate).


In [ ]:
import sys
import jc_wc
from matplotlib.lines import Line2D


def rabi_iv(dots, cavity, leads_template, lam, V_values):
    """One-sided-bias (mu_L=V, mu_R=0) I(V) for the Rabi model. Hamiltonian +
    photon decay built once; only the Fermi-dressed tunnel rates rebuilt per V."""
    leads = copy.deepcopy(leads_template)
    eigvals, eigvecs, _ = HamiltonianBuilder(cavity, dots, lam).diagonalize()
    Gamma_ph = CavityDecayMatrix(cavity, eigvals, eigvecs).build()
    transp = TransportCalculator(cavity, leads)
    I = np.empty(len(V_values))
    for j, V in enumerate(V_values):
        leads.mu_L, leads.mu_R = V, 0.0
        G_L, _G_R, G_tot = TunnelRateMatrix(cavity, dots, leads, eigvals, eigvecs).build()
        P = RateEquationSolver(G_tot, Gamma_ph, cavity).steady_solver()
        I[j] = transp.compute_current(G_L, P)
    return I


def rabi_thresholds(t, lam, strength_cut=0.02):
    """Per-panel Rabi step locations (raw energies, |0,0> at 0):
    (eps_g, [polariton energies], step_20). Polaritons = up to 2 brightest
    sector-1 states reachable from |0,0> besides the ground."""
    e = eps_g + t
    M = n_cut + 1
    dots = DotParameters(e1=e, e2=e, t=t)
    cav = CavityParameters(n=n_cut, omega=omega, kappa=1e-12)
    ev, evec, _ = HamiltonianBuilder(cav, dots, lam).diagonalize()
    probe = LeadParameters(gamma_L=1.0, gamma_R=1e-9, mu_L=50.0, mu_R=0.0, mu_0=0.0,
                           T_L=1e-3, T_R=1e-3, lam_L=0.0, lam_R=0.0)
    G_L, _G_R, _G_tot = TunnelRateMatrix(cav, dots, probe, ev, evec).build()
    i00 = int(np.argmin(np.abs(ev[:M])))
    s1 = list(range(M, 3 * M))
    energies = np.array([ev[f] for f in s1])
    strengths = np.array([G_L[i00, f] for f in s1])      # Dot_Class: Gamma[i,f]=rate i->f
    e_ground = energies.min()
    bright = [(en, s) for en, s in zip(energies, strengths)
              if en > e_ground + 1e-6 and s > strength_cut]
    bright.sort(key=lambda x: -x[1])
    pol = sorted(en for en, _ in bright[:2])
    return e_ground, pol, 2 * e - e_ground


# ---- comparison grid parameters ----
omega = 1.0
eps_g = 0.1                              # bonding ground kept in transport window
t_values = [0.4, 0.5, 0.6]              # Delta_omega = omega - 2t = +0.2, 0, -0.2
detuning_titles = [rf"$\Delta\omega={omega - 2*t:+.1f}$ ($t={t}$)" for t in t_values]
lam_values = [0.0, 0.2, 0.5]            # polaron lambda (Rabi); JC fed t*lambda
grid_kappas = [0.0, 1.0]                # the two fixed-kappa grids (sec 9a, 9b)
n_cut = 6
V = np.linspace(-0.5, 4.0, 800)         # step width ~ T (~0.004); raise N for sharper steps


def _kappa(k):
    return k if k > 0 else 1e-12        # strict 0 -> degenerate steady state


leads_rabi = LeadParameters(gamma_L=1e-3, gamma_R=1e-3, mu_L=0.0, mu_R=0.0,
                            mu_0=0.0, T_L=1e-3, T_R=1e-3, lam_L=0.0, lam_R=0.0)
leads_jc = jc_wc.LeadParameters(gamma_L=1e-3, gamma_R=1e-3, mu_L=0.0, mu_R=0.0,
                                mu_0=0.0, T_L=1e-3, T_R=1e-3, lambda_L=0.0, lambda_R=0.0)

# ---- compute every curve once: IV[(t, lam, kappa, model)] = I(V) ----
IV = {}
for t in t_values:
    e = eps_g + t
    dots = DotParameters(e1=e, e2=e, t=t)
    for lam in lam_values:
        for kappa in grid_kappas:
            cav_r = CavityParameters(n=n_cut, omega=omega, kappa=_kappa(kappa))
            cav_j = jc_wc.CavityParameters(n=n_cut, omega=omega, kappa=_kappa(kappa))
            IV[(t, lam, kappa, "Rabi")] = rabi_iv(dots, cav_r, leads_rabi, lam, V)
            IV[(t, lam, kappa, "JC")] = jc_wc.iv_curve(dots, cav_j, leads_jc, t * lam, V)
print(f"Computed {len(IV)} I-V curves over kappa in {grid_kappas}.")


def plot_iv_grid(kappa_plot):
    """Rabi (solid black) vs JC (dashed orange) I-V at a single fixed kappa.
    Rows = lam_values, columns = detunings. Per-panel Rabi thresholds (dotted
    vlines: eps_g/LP/UP/|2,0> step) and LP/UP zoom insets for lam in {0.2, 0.5}."""
    rabi_col, jc_col = "black", "tab:orange"
    fig, axes = plt.subplots(len(lam_values), len(t_values),
                             figsize=(15, 3.6 * len(lam_values)),
                             sharex=True, sharey=True, squeeze=False)
    for r, lam in enumerate(lam_values):
        for c, t in enumerate(t_values):
            ax = axes[r, c]
            ax.plot(V, IV[(t, lam, kappa_plot, "Rabi")], color=rabi_col, ls="-",  lw=1.3)
            ax.plot(V, IV[(t, lam, kappa_plot, "JC")],   color=jc_col,   ls="--", lw=1.3)
            e_g, pol, step20 = rabi_thresholds(t, lam)
            ax.axvline(e_g, color="0.4", ls=":", lw=1.0)
            for pe, pcol in zip(pol, ["tab:blue", "tab:red"]):
                ax.axvline(pe, color=pcol, ls=":", lw=1.0)
            ax.axvline(step20, color="tab:green", ls=":", lw=1.0)
            # second electron from the lower polariton: |LP> -> |2,1> at 2e + omega - E_LP.
            # Only a kappa->0 feature (the 1-photon |2,1> relaxes at finite kappa), so
            # the marker is drawn only on the kappa=0 grid.
            if len(pol) == 2 and np.isclose(kappa_plot, 0.0):
                lp21 = 2 * (eps_g + t) + omega - pol[0]
                ax.axvline(lp21, color="tab:purple", ls=":", lw=1.0)
            ax.grid(True, alpha=0.3)
            # LP<->UP zoom inset on a fine local grid (lambda in {0.2, 0.5})
            if any(np.isclose(lam, x) for x in (0.2, 0.5)) and len(pol) == 2:
                elp, eup = pol
                margin = 0.25 * (eup - elp)
                xlo, xhi = elp - margin, eup + margin
                axin = ax.inset_axes([0.62, 0.10, 0.34, 0.34])
                Vz = np.linspace(xlo, xhi, 400)
                e_dot = eps_g + t
                dots_z = DotParameters(e1=e_dot, e2=e_dot, t=t)
                cav_r = CavityParameters(n=n_cut, omega=omega, kappa=_kappa(kappa_plot))
                cav_j = jc_wc.CavityParameters(n=n_cut, omega=omega, kappa=_kappa(kappa_plot))
                Ir = rabi_iv(dots_z, cav_r, leads_rabi, lam, Vz)
                Ij = jc_wc.iv_curve(dots_z, cav_j, leads_jc, t * lam, Vz)
                axin.plot(Vz, Ir, color=rabi_col, ls="-",  lw=1.0)
                axin.plot(Vz, Ij, color=jc_col,   ls="--", lw=1.0)
                axin.axvline(elp, color="tab:blue", ls=":", lw=0.9)
                axin.axvline(eup, color="tab:red",  ls=":", lw=0.9)
                if np.isclose(kappa_plot, 0.0):
                    axin.axvline(2 * (eps_g + t) + omega - elp, color="tab:purple", ls=":", lw=0.9)
                yz = np.concatenate([Ir, Ij])
                axin.set_xlim(xlo, xhi)
                axin.set_ylim(yz.min() - 0.02, yz.max() + 0.02)
                axin.tick_params(labelsize=6)
                axin.set_title("LP/UP zoom", fontsize=6)
                v_mid = 0.5 * (elp + eup)
                i_mid = float(np.interp(v_mid, Vz, Ir))
                ax.annotate("", xy=(v_mid, i_mid), xycoords="data",
                            xytext=(0.62, 0.46), textcoords="axes fraction",
                            arrowprops=dict(arrowstyle="->", color="0.25", lw=1.1))
            if r == 0:
                ax.set_title(detuning_titles[c])
            if c == 0:
                ax.set_ylabel(rf"$\lambda={lam:g}$" + "\n\n" + r"$I$")
            if r == len(lam_values) - 1:
                ax.set_xlabel(r"$V_b$")
    curve_handles = [Line2D([0], [0], color=rabi_col, ls="-",  label="Rabi"),
                     Line2D([0], [0], color=jc_col,   ls="--", label=r"JC ($g=t\lambda$)")]
    thr_handles = [Line2D([0], [0], color="0.4",      ls=":", label=r"$\varepsilon_g$"),
                   Line2D([0], [0], color="tab:blue", ls=":", label="LP"),
                   Line2D([0], [0], color="tab:red",  ls=":", label="UP"),
                   Line2D([0], [0], color="tab:green", ls=":", label=r"$|2,0\rangle$ step")]
    if np.isclose(kappa_plot, 0.0):
        thr_handles.append(
            Line2D([0], [0], color="tab:purple", ls=":", label=r"$|LP\rangle\to|2,1\rangle$"))
    lead_ax = axes[0, -1]
    leg_main = lead_ax.legend(handles=curve_handles, loc="upper right", fontsize=8)
    lead_ax.add_artist(leg_main)
    lead_ax.legend(handles=thr_handles, loc="lower right", fontsize=7,
                   title="threshold $V_b$", title_fontsize=7)
    fig.suptitle(rf"Rabi vs JC I–V at fixed $\kappa={kappa_plot:g}$" 
                 "\n" 
                 r"(rows = $\lambda$, columns = detuning)", y=1.002)
    fig.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f"rabi_jc_iv_grid_kappa{kappa_plot:g}"+".png"), dpi=150, bbox_inches="tight"); plt.show()


### 9a. Fixed κ = 0 — rows = λ, columns = detuning


In [ ]:
plot_iv_grid(0.0)


### 9a — observations

Threshold markers (dotted vlines, computed per panel from the Rabi eigenvalues):
**gray = $\varepsilon_g$, blue = LP, red = UP, green = $|2,0\rangle$ step**. At
$\lambda=0$ the LP/UP markers collapse onto $\varepsilon_e$ (no splitting).

**1. LP/UP step-height asymmetry (e.g. resonance, $\lambda=0.2$).** A polariton
step's height is the injection oscillator strength from $|0,0\rangle$. The RWA
forces JC to split it evenly at resonance (LP/UP $=0.25/0.25$); the Rabi
counter-rotating terms tilt weight toward the LP ($0.287/0.213$). This small
mismatch is a genuine RWA-vs-full signature, already visible at $\lambda=0.2$.

**2. Higher polariton steps are $\lambda$-suppressed, not cut off.** Reaching the
$k$-photon polariton manifold from the empty dot is a $k$-photon process with
strength $\propto \lambda^{2k}$. At $\lambda=0.2$ the 2-photon strength is
$\sim10^{-3}$ (invisible) — the $n=6$ cutoff is *not* the limitation; the steps
grow with $\lambda$ and become visible by $\lambda=1$.

**3. The $|g,0\rangle$ threshold moves in Rabi but not in JC ($\lambda=1$).** The
full displacement dresses $|g,0\rangle$, renormalizing the inter-dot hopping by
the Franck–Condon factor $t_{\mathrm{eff}}\approx t\,e^{-\lambda^2/2}$, so
$\varepsilon_g=e-t_{\mathrm{eff}}$ rises ($0.10\to{\sim}0.20\text{–}0.23$ at
$\lambda=1$). JC linearizes the displacement and the RWA leaves $|g,0\rangle$
uncoupled, freezing it at $e-t=0.10$. A genuine strong-coupling (polaron) effect
that JC structurally misses.

**4. The dip in the JC curve (below resonance, $\lambda=1$) is a polariton
trap.** It sits at $V\approx E_{\mathrm{LP}}$, the lower polariton, which below
resonance is photon-dominated ($\langle n\rangle\approx0.58$). Once $V$ opens the
$|0,0\rangle\to\mathrm{LP}$ channel, the photon-like LP cannot tunnel out (leads
do not change photon number) and at $\kappa\to0$ the photon cannot decay — so
population traps there ($P:\,0\to\tfrac13$) and the current drops (NDR). It
should weaken/vanish at finite $\kappa$ (the $\kappa=0.1,\,1$ rows).

**5. Extra step between $|2,0\rangle$ and UP — second electron from the
polariton (purple marker).** Once the lower polariton is occupied, a *second*
electron can enter it, $|\mathrm{LP}\rangle\to|2,1\rangle$ (doubly-occupied dot
with one cavity photon), at $V = 2e + \omega - E_{\mathrm{LP}}$ (purple dotted
line; $\approx 1.32$ at resonance, $\lambda=0.5$). The other markers cover only
first-electron thresholds plus $|g,0\rangle\to|2,0\rangle$, so this transition
was previously unlabelled. It is a $\kappa\to0$ feature: at finite $\kappa$ the
one-photon $|2,1\rangle$ relaxes to $|2,0\rangle$, so the step is present here
(§9a) but suppressed in §9b — the purple marker in §9b shows where it *would* sit.

**Convention note.** The Rabi (`Dot_Class`) classes store
$\Gamma[i,f]=\mathrm{rate}(i\to f)$ while the JC (`jc_wc`) classes store
$\Gamma[f,i]$. Within the Rabi model *both* the tunnel and cavity-decay matrices
must use $\Gamma[i,f]$ (a bug where `CavityDecayMatrix` used $\Gamma[f,i]$ made
the decay enter the master equation transposed — photon *absorption* instead of
emission — pumping the cavity to the Fock cutoff and blocking the current at
finite $\kappa$; now fixed). Do not mix the two models' internals.


### 9b. Fixed κ = 1 — rows = λ, columns = detuning


In [ ]:
plot_iv_grid(1.0)


## 10. Sweep helper and its applications

`iv_curve_sweep` builds the Hamiltonian, photon-decay matrix, and transport
calculator **once per `lam`**; only the Fermi-dressed tunneling rates are
rebuilt inside the V loop. §10.2–10.3 are special cases of this one helper.

### 10.1. The `iv_curve_sweep` helper


In [ ]:
def iv_curve_sweep(dots, cavity, leads_template, V_array,
                   *, lam_values, kappa, dress_leads=True):
    
    """Return I(V) for each lam in lam_values, at fixed kappa.

    Builds HamiltonianBuilder + CavityDecayMatrix + TransportCalculator
    once per lam (since they don't depend on V); only TunnelRateMatrix
    is rebuilt inside the V loop because it depends on mu_L, mu_R.

    Parameters
    ----------
    dots, cavity, leads_template : the three parameter objects.
        `cavity.kappa` is overwritten with `kappa` argument.
    V_array : 1-D array of bias values to scan.
    lam_values : iterable of electron-photon couplings.
    kappa : float, cavity decay rate to use for this sweep.
    dress_leads : if True, also set leads.lam_L = leads.lam_R = lam.

    Returns
    -------
    dict {lam: np.ndarray of shape V_array.shape}
    """
    
    cavity.kappa = kappa
    results = {}
    for lam in lam_values:
        # Per-lam: Hamiltonian, eigenbasis, photon decay, transport
        ham = HamiltonianBuilder(cavity, dots, lam)
        eigvals, eigvecs, _ = ham.diagonalize()
        Gamma_ph = CavityDecayMatrix(cavity, eigvals, eigvecs).build()
        transport = TransportCalculator(cavity, leads_template)

        currents = np.empty(len(V_array))
        for k, V in enumerate(V_array):
            leads = copy.deepcopy(leads_template)
            leads.mu_L = leads.mu_0 + V
            leads.mu_R = leads.mu_0 
            if dress_leads:
                leads.lam_L = lam
                leads.lam_R = lam

            G_L, _G_R, G_tot = TunnelRateMatrix(
                cavity, dots, leads, eigvals, eigvecs
            ).build()

            P = RateEquationSolver(G_tot, Gamma_ph, cavity).steady_solver()
            currents[k] = transport.compute_current(G_L, P)

        results[lam] = currents
    return results


### 10.2. λ-sweep vs κ at three detunings

Rabi I–V grid: **columns = resonance condition** (bright bonding with $\varepsilon_g$ fixed at 0.1 via `e1=e2=0.1+t`,
`t ∈ {0.4, 0.5, 0.6}` ⇒ Δω = ω − 2t = {+0.2, 0, −0.2}), **rows = cavity decay
`κ ∈ {0, 0.1, 1}`**. Inside each panel λ ∈ {0, 0.2, 0.5, 1} is swept; **λ = 0
(black dashed) is the non-interacting reference** (cavity decoupled, so it is
identical in every row). Comparing across a column shows the effect of κ; within
a panel, the effect of λ. (κ = 0 uses a 1e-12 proxy; one-sided bias μ_L=V, μ_R=0.)


In [ ]:
# 10.2: rows = kappa, columns = detuning (resonance condition); sweep lambda inside.
# lambda = 0 is the non-interacting reference (cavity decoupled => same in every row).
eps_g = 0.1                                      # bonding |g,0> kept in transport window (like 9a)
omega = 1.0
t_values = [0.4, 0.5, 0.6]                       # Delta_w = omega - 2t = +0.2, 0, -0.2
detuning_labels = [r"$\Delta\omega=+0.2$ ($t=0.4$)",
                   r"$\Delta\omega=0$ ($t=0.5$)",
                   r"$\Delta\omega=-0.2$ ($t=0.6$)"]
lam_values = [0.0, 0.5, 1.0, 1.5, 2]
kappa_rows = [0.0, 0.1, 1.0]
n_phot = 6
V_values = np.linspace(-2.0, 3.0, 600)           # fine grid -> sharp steps


def _kap(k):
    return k if k > 0 else 1e-12                 # strict 0 -> degenerate steady state


leads_template = LeadParameters(
    gamma_L=0.001, gamma_R=0.001, mu_L=0.0, mu_R=0.0, mu_0=0.0,
    T_L=0.001, T_R=0.001, lam_L=0.0, lam_R=0.0,
)

fig, axes = plt.subplots(len(kappa_rows), len(t_values),
                         figsize=(15, 4.0 * len(kappa_rows)),
                         sharex=True, sharey=True, squeeze=False)
lam_pos = [l for l in lam_values if l > 0]
colors = plt.cm.viridis(np.linspace(0.2, 0.85, len(lam_pos)))

for r, kappa in enumerate(kappa_rows):
    for c, t in enumerate(t_values):
        ax = axes[r, c]
        e = eps_g + t                            # e1 = e2 = 0.1 + t  =>  eps_g = 0.1 for every column
        dots = DotParameters(e1=e, e2=e, t=t)
        cavity = CavityParameters(n=n_phot, omega=omega, kappa=_kap(kappa))
        curves = iv_curve_sweep(dots, cavity, leads_template, V_values,
                                lam_values=lam_values, kappa=_kap(kappa),
                                dress_leads=False)
        # non-interacting reference (lambda = 0), identical for every kappa
        ax.plot(V_values, curves[0.0], color="orange", ls="--", lw=1.1,
                label=r"$\lambda=0$ (ref)")
        for col, lam in zip(colors, lam_pos):
            ax.plot(V_values, curves[lam], color=col, lw=1.3,
                    label=rf"$\lambda={lam:g}$")
        ax.grid(True, alpha=0.3)
        if r == 0:
            ax.set_title(detuning_labels[c])
        if c == 0:
            ax.set_ylabel(rf"$\kappa={kappa:g}$" + "\n\n" + r"$I$")
        if r == len(kappa_rows) - 1:
            ax.set_xlabel(r"$V_b$")

axes[0, -1].legend(loc="lower right", fontsize=8)
fig.suptitle(r"Rabi I–V ($\varepsilon_g=0.1$ fixed) — rows = $\kappa$, columns = detuning, "
             r"$\lambda$ swept within each panel ($\lambda=0$ = non-interacting reference)", y=1.005)
fig.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "lambda_sweep_vs_kappa_grid"+".png"), dpi=150, bbox_inches="tight"); plt.show()


### 10.3. κ-sweep at three detunings

Same Δω panels (bright bonding, $\varepsilon_g=0.1$ fixed via `e1=e2=0.1+t`); sweep κ ∈ {10⁻⁴, 10⁻³, 10⁻², 10⁻¹} at λ=0.5, with the λ=κ=0 reference.


In [ ]:
# Same Delta_w panels, sweep kappa at fixed lambda.
eps_g = 0.1                              # bonding |g,0> kept in transport window (like 9a)
omega = 1.0
t_values = [0.4, 0.5, 0.6]
detuning_labels = [r"$\Delta\omega=+0.2$",
                   r"$\Delta\omega=0$",
                   r"$\Delta\omega=-0.2$"]
kappa_values = [1e-4, 1e-3, 1e-2, 1e-1]
lam_main = 0.5
V_values = np.linspace(-2.0, 3.0, 200)

leads_template = LeadParameters(
    gamma_L=0.002, gamma_R=0.002,
    mu_L=0.0, mu_R=0.0, mu_0=0.0,
    T_L=0.001, T_R=0.001,
    lam_L=0.0, lam_R=0.0,
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(kappa_values)))

for ax, t, dlabel in zip(axes, t_values, detuning_labels):
    e = eps_g + t                            # e1 = e2 = 0.1 + t  =>  eps_g = 0.1 for every column
    dots = DotParameters(e1=e, e2=e, t=t)
    cavity = CavityParameters(n=8, omega=omega, kappa=kappa_values[0])

    # lam=kappa=0 reference
    ref = iv_curve_sweep(
        dots, cavity, leads_template, V_values,
        lam_values=[0.0], kappa=0.0, dress_leads=False,
    )[0.0]
    ax.plot(V_values, ref, color="black", linestyle="--", lw=1.0,
            label=r"$\lambda=\kappa=0$")

    for c, kappa in zip(colors, kappa_values):
        curves = iv_curve_sweep(
            dots, cavity, leads_template, V_values,
            lam_values=[lam_main], kappa=kappa, dress_leads=False,
        )
        ax.plot(V_values, curves[lam_main], color=c, lw=1.2,
                label=rf"$\kappa={kappa:g}$")

    ax.set_xlabel(r"$V_b$")
    ax.set_title(dlabel + rf",  $t={t}$")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel(r"$I$")
axes[-1].legend(loc="lower right", fontsize=8)
fig.suptitle(rf"$\kappa$-sweep at three detunings, $\lambda={lam_main}$",
             y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "kappa_sweep_three_detunings"+".png"), dpi=150, bbox_inches="tight"); plt.show()


## 11. Example — single IV sweep (smoke test)

One bright-bonding IV curve at `e1=e2=0.6, t=0.5, ω=1, λ=0.5, κ=10⁻³`. End-to-end check that all classes wire together.


In [ ]:
# Bright-bonding smoke test: ε_g = +0.1 puts the bonding singlet above
# zero, so |0,0> can empty into the leads and the LP/UP/ε_e steps are
# clearly visible.

dots = DotParameters(e1=0.6, e2=0.6, t=0.5)
cavity = CavityParameters(n=8, omega=1.0, kappa=1e-3)
leads_template = LeadParameters(
    gamma_L=0.002, gamma_R=0.002,
    mu_L=0.0, mu_R=0.0, mu_0=0.0,
    T_L=0.001, T_R=0.001,
    lam_L=0.0, lam_R=0.0,
)

V_values = np.linspace(-2.0, 3.0, 200)

curves = iv_curve_sweep(
    dots, cavity, leads_template, V_values,
    lam_values=[0.5], kappa=1e-3, dress_leads=False,
)
I = curves[0.5]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(V_values, I, lw=1.2, label=r"$\lambda=0.5,\ \kappa=10^{-3}$")
ax.set_xlabel(r"$V_b$")
ax.set_ylabel(r"$I$")
ax.set_title(r"Bright-bonding IV — single-curve smoke test")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "smoke_test_single_iv"+".png"), dpi=150, bbox_inches="tight"); plt.show()


## 12. Franck–Condon lead-dressing (κ = 1)

> **Notation.** A tilde marks a *single-charge eigenstate* (a polaron): the ground is
> $|\tilde g,0\rangle$ and the $n$-excitation polariton doublets are $|n^\mp\rangle$ (= LP$_n$/UP$_n$).
> Plain kets are *bare products* — the orbital$\otimes$Fock basis $|g,m\rangle,|e,m\rangle$ used for
> decompositions, plus the empty/double sectors $|0,m\rangle,|2,m\rangle$ (exact eigenstates, since the
> dressed hopping acts only in the single-charge sector). Thus $|\tilde g,0\rangle\approx|g,0\rangle$ at small $\lambda$.

Turning on the lead dressings $\lambda_L,\lambda_R$ puts a
displacement $\hat D(\lambda_\ell)$ on each lead–dot tunnel vertex, so every time an
electron hops between a lead and the dots it **shakes the cavity**. The elastic
(0-photon) tunnel element through a dressed lead carries the Franck–Condon overlap
$\langle 0|\hat D(\lambda_\ell)|0\rangle = e^{-\lambda_\ell^2/2}$, so the first (elastic)
plateau is **suppressed**, while extra steps spaced by $\omega$ — the photon sidebands —
reopen current at higher bias.

**What the sidebands actually are (charge selection rule).** A lead vertex carries a
fermionic operator $d_\ell^\dagger$ (charging) or $d_\ell$ (discharging) and changes the dot
occupation by exactly $\pm1$, so every transport transition connects **adjacent charge
sectors**. The single-charge eigenstates ($|\tilde g,0\rangle$ and the doublets LP/UP) all live in the *single-charge*
sector, so there is **no** transport hop among them — in particular none like
$|\tilde g,0\rangle\!\to\!|g,1\rangle$ or $|\tilde g,0\rangle\!\to\!\mathrm{LP/UP}$. The elastic plateau is the
charging cycle $|0,0\rangle\!\to\!|\tilde g,0\rangle\!\to\!|0,0\rangle$ into the lowest polariton
$|\tilde g,0\rangle$ ($\approx|g,0\rangle$ bare); the $\omega$-spaced recovery steps are the *higher* charging
transitions $|0,0\rangle\!\to\!\mathrm{LP/UP}$, with the extra photon supplied by the
$\langle1|\hat D(\lambda_\ell)|0\rangle$ piece of the vertex — not by a hop within the
single-charge manifold. Moreover each lead couples to a **local** dot,
$d_1=\tfrac1{\sqrt2}(d_g+d_e)$, $d_2=\tfrac1{\sqrt2}(d_g-d_e)$, so injection from $|0,0\rangle$
creates a $(d_g^\dagger\pm d_e^\dagger)$ superposition that overlaps **both** LP and UP. The
lead-dressing factor is therefore the polariton-weighted sum of
$\langle n'|\hat D(\lambda_\ell)|n\rangle$, collapsing to the single $e^{-\lambda_\ell^2/2}$ only
when the target branch is nearly pure $|g,0\rangle$.

Here $\lambda$ is the **inter-dot** dressing (`HamiltonianBuilder`) and $\lambda_L,\lambda_R$
the **lead** dressings (`LeadParameters`). We compare three switch-on scenarios
$(\lambda,\lambda_L,\lambda_R)\in\{(1,1,0),(1,0,1),(1,1,1)\}$ at $\kappa=1$ (so the sideband
photons relax) against the undressed, undriven reference $\lambda=0,\ \kappa=0$.

Because the plateau current is normalized by $\gamma_L\gamma_R/(\gamma_L+\gamma_R)$, the
reference elastic plateau sits at $\approx 0.5$ and the dressed plateaus collapse toward
$\approx 0.5\,e^{-(\lambda_L^2+\lambda_R^2)/2}\cdot(\dots)$ — a single bottleneck lead in
$(1,1,0)/(1,0,1)$, both leads in $(1,1,1)$. The inter-dot $\lambda=1$ also renormalizes the
hopping $t_{\rm eff}=t\,e^{-\lambda^2/2}$, pushing $\varepsilon_g=e-t_{\rm eff}$ to higher bias
(the polaron shift), so the dressed first steps sit to the right of the reference.

In [ ]:
# 12. Franck-Condon lead-dressing: independent inter-dot (lam) and lead (lam_L, lam_R)
# displacements.  Built on the same one-sided-bias driver as sec 9, but with the lead
# dressings set explicitly so we can switch them on asymmetrically.

def rabi_iv_fc(dots, cavity, leads_template, lam, lam_L, lam_R, V_values):
    """One-sided-bias (mu_L = V, mu_R = 0) I(V) with independent inter-dot dressing
    `lam` and lead dressings `lam_L`, `lam_R`.  H + photon decay are built once; only
    the Fermi-dressed lead tunnel rates are rebuilt per bias point."""
    leads = copy.deepcopy(leads_template)
    leads.lam_L, leads.lam_R = lam_L, lam_R
    eigvals, eigvecs, _ = HamiltonianBuilder(cavity, dots, lam).diagonalize()
    Gamma_ph = CavityDecayMatrix(cavity, eigvals, eigvecs).build()
    transp = TransportCalculator(cavity, leads)
    I = np.empty(len(V_values))
    for j, V in enumerate(V_values):
        leads.mu_L, leads.mu_R = V, 0.0
        G_L, _G_R, G_tot = TunnelRateMatrix(cavity, dots, leads, eigvals, eigvecs).build()
        P = RateEquationSolver(G_tot, Gamma_ph, cavity).steady_solver()
        I[j] = transp.compute_current(G_L, P)
    return I


# ---- parameters: resonance (Delta_omega = 0), bright bonding (conventions of sec 9/10) ----
omega_fc = 1.0
eps_g_fc = 0.1
t_fc     = 0.5                                   # Delta_omega = omega - 2t = 0
e_fc     = eps_g_fc + t_fc                        # e1 = e2 so eps_g (bare) = 0.1
n_fc     = 10                                     # Fock cutoff N (= max photon number)
V_fc     = np.linspace(-0.5, 4.0, 800)            # 800 pts -> sharp steps

dots_fc  = DotParameters(e1=e_fc, e2=e_fc, t=t_fc)
leads_fc = LeadParameters(gamma_L=1e-3, gamma_R=1e-3, mu_L=0.0, mu_R=0.0, mu_0=0.0,
                          T_L=1e-3, T_R=1e-3, lam_L=0.0, lam_R=0.0)

# (lam, lam_L, lam_R) scenarios at kappa = 1.  (1,0,0) = interacting (inter-dot lam=1) but
# BARE leads -> isolates the effect of the LEAD dressing in the other three.
scenarios  = [(1, 0, 0), (1.0, 1.0, 0.0), (1.0, 0.0, 1.0), (1.0, 1.0, 1.0)]
colors_fc  = ["tab:orange", "tab:blue", "tab:green", "tab:red"]

cav_ref = CavityParameters(n=n_fc, omega=omega_fc, kappa=1e-12)   # strict 0 -> degenerate
cav_fc  = CavityParameters(n=n_fc, omega=omega_fc, kappa=1.0)

# I(V): undressed reference + the four kappa=1 scenarios
I_ref = rabi_iv_fc(dots_fc, cav_ref, leads_fc, 0.0, 0.0, 0.0, V_fc)
I_scn = {sc: rabi_iv_fc(dots_fc, cav_fc, leads_fc, *sc, V_fc) for sc in scenarios}

# ================== channel-resolved FULL rates (Gamma_L+Gamma_R)/gamma_L ==================
# Each I-V step is a charging transition switching on at its Fermi edge V = Delta_E.  The
# single-charge sector is diagonalised ONCE (the inter-dot lam=1 Hamiltonian is the same for
# every scenario; only the lead dressing differs), giving eigenstates labelled by excitation
# manifold: ground |g,0>, then polariton doublets |n-> (LPn) / |n+> (UPn).  Double-charge
# states are bare products |2,m> (doubly-occupied dot, m photons), like the empty sector |0,m>.

ev_m, evec_m, _ = HamiltonianBuilder(cav_fc, dots_fc, 1.0).diagonalize()
M_m  = n_fc + 1
s1_m = list(range(M_m, 3 * M_m)); s2_m = list(range(3 * M_m, 4 * M_m))
i00  = int(np.argmin(np.abs(ev_m[:M_m])))                       # |0,0>
ig   = min(s1_m, key=lambda f: ev_m[f])                         # |g,0>
sec1_sorted = sorted(s1_m, key=lambda f: ev_m[f])
sec2_sorted = sorted(s2_m, key=lambda f: ev_m[f])
gL = leads_fc.gamma_L

def sec1_label(k):                                              # |g,0>, |1->, |1+>, |2->, |2+>, ...
    if k == 0:
        return r"$|g,0\rangle$"
    n, sign = (k + 1) // 2, ("-" if k % 2 == 1 else "+")
    return rf"$|{n}^{{{sign}}}\rangle$"

def total_rate_rows(lam_L, lam_R, init_idx):
    """(Gamma_L+Gamma_R)[init_idx, :] / gamma_L vs V, for lead dressing (lam_L, lam_R)."""
    leads = copy.deepcopy(leads_fc); leads.lam_L, leads.lam_R = lam_L, lam_R
    R = np.empty((len(V_fc), 4 * M_m))
    for k, V in enumerate(V_fc):
        leads.mu_L, leads.mu_R = V, 0.0
        GL, GR, _ = TunnelRateMatrix(cav_fc, dots_fc, leads, ev_m, evec_m).build()
        R[k] = (GL[init_idx] + GR[init_idx]) / gL
    return R

# (1,1,0) dressed-LEFT: 1st-/2nd-electron rows ;  (1,0,0) interacting bare-leads: both rows
R00_d = total_rate_rows(1.0, 0.0, i00)
Rg0_d = total_rate_rows(1.0, 0.0, ig)
R00_i = total_rate_rows(0.0, 0.0, i00)
Rg0_i = total_rate_rows(0.0, 0.0, ig)

CUT = 5e-3                               # below this a channel is treated as identically zero

def plot_rates(ax, Rmat, idx_sorted, label_fn, ls="-", c0=0):
    """Plot total rate vs V; colour+label activating channels, gray the dark ones.
    Returns (activation list, next colour index)."""
    activ, ci = [], c0
    cmap = plt.cm.tab10
    for k, f in enumerate(idx_sorted):
        tr = Rmat[:, f]
        if tr.max() > CUT:
            ax.plot(V_fc, tr, color=cmap(ci % 10), lw=1.6, ls=ls, label=label_fn(k))
            activ.append((label_fn(k), float(tr.max()))); ci += 1
        else:
            ax.plot(V_fc, tr, color="0.85", lw=0.6, ls=ls, zorder=0)
    return activ, ci

# ================== figure: I-V, then (1,1,0) 1st/2nd-e rates, then (1,0,0) all rates ==================
fig, (axI, axA, axB, axC) = plt.subplots(4, 1, figsize=(9.5, 15), sharex=True,
                                         gridspec_kw=dict(height_ratios=[1.3, 1, 1, 1.15]))

axI.plot(V_fc, I_ref, color="black", ls="--", lw=1.5, label=r"ref: $\lambda=0,\ \kappa=0$")
for sc, col in zip(scenarios, colors_fc):
    lam, lL, lR = sc
    axI.plot(V_fc, I_scn[sc], color=col, lw=1.5,
             label=rf"$(\lambda,\lambda_L,\lambda_R)=({lam:g},{lL:g},{lR:g}),\ \kappa=1$")
axI.set_ylabel(r"$I$")
axI.set_title(r"Franck-Condon lead-dressing at $\kappa=1$ — I-V and channel activation "
              "\n"
              r"($t=0.5$, $\Delta\omega=0$, bright bonding $\varepsilon_g=0.1$, $N=10$)")
axI.grid(True, alpha=0.3); axI.legend(loc="upper left", fontsize=8)

plot_rates(axA, R00_d, sec1_sorted, sec1_label)
axA.set_ylabel(r"$(\Gamma_L+\Gamma_R)/\gamma_L$")
axA.set_title(r"DRESSED $(1,1,0)$ — 1st electron:  $|0,0\rangle \to$ single-charge eigenstates",
              fontsize=10)
axA.grid(True, alpha=0.3); axA.legend(loc="upper left", fontsize=8, ncol=2)

plot_rates(axB, Rg0_d, sec2_sorted, lambda k: rf"$|2,{k}\rangle$")
axB.set_ylabel(r"$(\Gamma_L+\Gamma_R)/\gamma_L$")
axB.set_title(r"DRESSED $(1,1,0)$ — 2nd electron:  $|g,0\rangle \to |2,m\rangle$", fontsize=10)
axB.grid(True, alpha=0.3); axB.legend(loc="upper left", fontsize=8, ncol=2)

# ---- (1,0,0): ALL rates in ONE panel.  solid = 1st electron, dashed = 2nd electron ----
a1, ci = plot_rates(axC, R00_i, sec1_sorted, sec1_label, ls="-",  c0=0)
a2, _  = plot_rates(axC, Rg0_i, sec2_sorted, lambda k: rf"$|2,{k}\rangle$", ls="--", c0=ci)
axC.set_ylabel(r"$(\Gamma_L+\Gamma_R)/\gamma_L$")
axC.set_title(r"INTERACTING, bare leads $(1,0,0)$ — ALL rates "
              r"(solid: $|0,0\rangle\to$single,  dashed: $|g,0\rangle\to|2,m\rangle$)", fontsize=10)
axC.grid(True, alpha=0.3); axC.legend(loc="upper left", fontsize=8, ncol=3)
axC.set_xlabel(r"$V_b$")

fig.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "franck_condon_lead_dressing"+".png"), dpi=150, bbox_inches="tight"); plt.show()

# ---- numeric summary: sum rule + activation heights ----
print("INTERACTING (1,0,0)  -- bare leads conserve injection weight (sum rule):")
print("   1st e- |0,0>->single :", [(l, round(h, 3)) for l, h in a1],
      " sum=%.3f" % sum(h for _, h in a1))
print("   2nd e- |g,0>->|2,m>  :", [(l, round(h, 3)) for l, h in a2])


### 12 — observations: lead Franck–Condon vs inter-dot dressing

All four interacting cases share the **same** $\lambda=1$ polariton spectrum, so the I–V
steps sit at the same voltages; only the **step heights** differ. Verified first-plateau
currents: ref $0.500\to(1,0,0)\,0.478\to(1,1,0)\,0.354\to(1,0,1)\,0.196\to(1,1,1)\,0.171$.

**1. The suppression is a LEAD effect, not the interaction.** The control $(1,0,0)$ (interacting
but **bare** leads) is essentially unsuppressed ($0.478\approx$ bare $0.5$). A bare vertex
$d_\ell^\dagger$ is photon-conserving and obeys the sum rule
$\sum_\alpha|\langle\alpha|d_L^\dagger|0,0\rangle|^2=\langle0,0|d_Ld_L^\dagger|0,0\rangle=1$; the
inter-dot dressing merely **redistributes** that unit weight among the *low-energy* 0-photon
polaritons — $|\tilde g,0\rangle$ ($0.478$) $+\,|1^-\rangle$ ($0.414$) $\approx$ full, both open by
$V\approx0.73$ — so the current saturates near unity by $V\approx1.3$. The dressed vertex
$d_\ell^\dagger\hat D(\lambda_\ell)$ obeys the *same* sum rule but scatters the unit weight into
**high-bias photon sidebands**, throttling the elastic line by $e^{-\lambda^2/2}$ and depleting
the low-bias channels — that depletion **is** the Franck–Condon blockade.

**2. $|1^-\rangle$ is the vacuum-Rabi-split antibonding.** With bare leads the large second step
($V=0.73$) is injection into $|1^-\rangle$, which carries the bare antibonding-0 ($|e,0\rangle$)
weight; the inter-dot coupling splits the $\{|e,0\rangle,|g,1\rangle\}$ pair into the Rabi doublet $|1^-\rangle/|2^-\rangle$ (both $P{=}+1$; full bare-basis decomposition in §6.1).
(In the dressed panels $|1^-\rangle$ is nearly dark — the dressed vertex weights the manifolds by
FC factors instead, making the *one-photon* polariton $|2^-\rangle$ the bright sideband.)

**3. Left/right asymmetry, $(1,1,0)\neq(1,0,1)$.** With one dressed lead the plateaus still differ
($0.354$ vs $0.196$). The inter-dot term $\mathfrak t\,e^{\lambda(a-a^\dagger)}d_1^\dagger d_2$ is
invariant only under the **combined** $(1\!\leftrightarrow\!2)\times(a\to-a)$, under which
$\hat D(\lambda)\to\hat D(-\lambda)$; so dressing the left lead with $+\lambda$ equals dressing the
right with $-\lambda$, **not** $+\lambda$. The dressed elastic overlap is $0.281$ (left) vs $0.123$
(right) — constructive vs destructive $b_m$–$a_m$ interference in the polaron eigenstate.

**4. Directionality under one-sided bias.** Only $\mu_L=V$ sweeps. Dressing the **injection** lead
$(1,1,0)$ opens FC sidebands as $V$ grows → blockade lifts ($I\to0.94$). Dressing the **extraction**
lead $(1,0,1)$ — whose sidebands $|\tilde g,0\rangle\to|0,m\rangle$ need the electron to leave below
$\mu_R=0$ (Pauli-blocked) — stays frozen at the elastic rate → saturates $\approx0.52$ at all bias;
$(1,1,1)$ inherits that extraction ceiling. $\kappa=1$ relaxes each emitted photon between cycles,
keeping the steps sharp and monotone (at $\kappa\to0$ these would trap → NDR, cf. §9a).

In [ ]:
# 12 — numerical check: the elastic |0,0> -> |g,0> suppression IS the Franck-Condon factor.
# Elastic injection rate / gamma = |<g,0| d_ell^dag D(lam_ell) |0,0>|^2.  If |g,0> were a pure
# product |g>(x)|0>, this equals 0.5 * e^{-lam_ell^2} (the bare FC factor, x 1/2 lead overlap).
# Verify that exactly in the product limit (inter-dot lam=0), then quantify the polaron
# correction at inter-dot lam=1 (the (1,1,0)/(1,0,1) scenarios).  Reuses sec-10 dots_fc/cav_fc.

def fc_factor_check():
    def elastic(lam_inter, lam_L, lam_R):
        """|W|^2 / gamma for the elastic |0,0> -> |g,0> step.  Reads the LEFT injection entry
        (high mu_L, f_L=1) when the left lead is the dressed/probed one, else the RIGHT
        extraction entry (mu_R=0, 1-f_R=1)."""
        ev, evec, _ = HamiltonianBuilder(cav_fc, dots_fc, lam_inter).diagonalize()
        M = n_fc + 1; s1 = range(M, 3 * M)
        i0 = int(np.argmin(np.abs(ev[:M])))          # |0,0>
        jg = min(s1, key=lambda f: ev[f])            # |g,0>
        leads = LeadParameters(gamma_L=1.0, gamma_R=1.0, mu_L=99.0, mu_R=0.0, mu_0=0.0,
                               T_L=1e-3, T_R=1e-3, lam_L=lam_L, lam_R=lam_R)
        GL, GR, _ = TunnelRateMatrix(cav_fc, dots_fc, leads, ev, evec).build()
        return GL[i0, jg] if lam_R == 0.0 else GR[jg, i0]

    print("elastic |0,0> -> |g,0>   (Gamma/gamma);   FC prediction = 0.5 * exp(-lam^2)\n")
    print("(1) CLEAN product limit: inter-dot lam = 0  ->  eigenstate is pure |g,0>")
    print(f"   {'lam_L':>6} {'Gamma/gamma':>12} {'0.5 e^-lamL^2':>14} {'ratio':>8}")
    for lL in [0.0, 0.5, 1.0, 1.5, 2.0]:
        r = elastic(0.0, lL, 0.0); fc = 0.5 * np.exp(-lL**2)
        print(f"   {lL:>6} {r:>12.5f} {fc:>14.5f} {r/fc:>8.4f}")

    print("\n(2) POLARON regime: inter-dot lam = 1   (the (1,1,0)/(1,0,1) scenarios)")
    b  = elastic(1.0, 0.0, 0.0)                      # undressed  (= (1,0,0))
    dL = elastic(1.0, 1.0, 0.0)                      # left  dressed (1,1,0)
    dR = elastic(1.0, 0.0, 1.0)                      # right dressed (1,0,1)
    print(f"   undressed  |W|^2              = {b:.4f}")
    print(f"   left  dressed (lam_L=1)       = {dL:.4f}    ratio dL/b = {dL/b:.4f}")
    print(f"   right dressed (lam_R=1)       = {dR:.4f}    ratio dR/b = {dR/b:.4f}")
    print(f"   pure FC factor  e^-lam^2      = {np.exp(-1.0):.4f}")
    print(f"   geometric mean  sqrt(dL dR)/b = {np.sqrt(dL * dR) / b:.4f}"
          "   (+/- polaron interference cancels -> recovers e^-1)")

fc_factor_check()


### 12 — first-step height and the Franck–Condon factor

The height of the **first** I–V step (first plateau) of $(1,1,0)$ is $I_1\approx0.354$. It is a
two-rate **series** — inject from the left, extract to the right (with $\gamma_L=\gamma_R$):

$$I_1=\frac{2\,R_{\rm inj}R_{\rm ext}}{R_{\rm inj}+R_{\rm ext}},\qquad
R_{\rm inj}=|W_L|^2=0.281,\quad R_{\rm ext}=|W_R|^2=0.478 .$$

The FC factor lives in the **amplitude** of the dressed injection vertex,
$W_L=\langle \tilde g,0|\,d_L^\dagger\hat D(\lambda_L)\,|0,0\rangle\to\tfrac{1}{\sqrt2}e^{-\lambda_L^2/2}$
in the product limit, so the injection **rate** carries it squared,
$R_{\rm inj}=\tfrac12 e^{-\lambda_L^2}\times(\text{polaron form factor})$; the extraction leg is
bare ($\lambda_R=0$) and carries **no** FC factor.

**Closed form (product limit, no polaron mixing).** With $R_{\rm inj}=\tfrac12 e^{-\lambda_L^2}$
and $R_{\rm ext}=\tfrac12$ the harmonic mean collapses to a clean function of the FC factor alone:

$$I_1^{\rm prod} = \frac{e^{-\lambda_L^2}}{1+e^{-\lambda_L^2}}\;\xrightarrow{\ \lambda_L=1\ }\;
\frac{0.368}{1.368} = 0.269 .$$

**Actual $(1,1,0)$, inter-dot $\lambda=1$.** The polaron eigenstate boosts $R_{\rm inj}$ from the
clean $\tfrac12 e^{-1}=0.184$ to $0.281$ (form factor $1.53$, cf. the `fc_factor_check` cell) and
trims $R_{\rm ext}$ from $0.5$ to $0.478$, giving
$I_1=2(0.281)(0.478)/(0.281+0.478)=\mathbf{0.354}$.

**Takeaway.** The first-step drop $0.500\to0.354$ is the FC factor $e^{-\lambda_L^2/2}$ acting
(squared, as a rate) on the **one dressed leg**, harmonic-mean–combined with the unsuppressed
bare extraction — which is why it is *not* simply $e^{-\lambda_L^2}$. Strip the inter-dot polaron
mixing and it is exactly $e^{-\lambda_L^2}/(1+e^{-\lambda_L^2})$; in the injection-bottleneck limit
($R_{\rm inj}\ll R_{\rm ext}$, larger $\lambda_L$) it reduces to $I_1\to 2R_{\rm inj}\propto
e^{-\lambda_L^2}$, tracking the FC factor directly.

### 12 — derivation of the first-plateau current (harmonic-mean formula)

The $I_1=2R_{\rm inj}R_{\rm ext}/(R_{\rm inj}+R_{\rm ext})$ used above is the standard single-level
sequential-tunneling current, specialised to the first plateau and put through the notebook's
current normalisation. Step by step:

**1. Two-state reduction.** For $\varepsilon_g<V<$ (next threshold $\approx0.73$) only
$|0,0\rangle$ (state $0$: empty dot + cavity vacuum) and $|\tilde g,0\rangle$ (state $1$) lie in the bias
window; everything else is energetically closed and $\kappa=1$ keeps the cavity in vacuum. So it is
a one-level dot between two leads.

**2. Which lead processes survive.** A lead $\ell$ drives $0\leftrightarrow1$ at rate
$\gamma_\ell|W_\ell|^2$ times a Fermi factor evaluated at $\varepsilon_g$. With $\mu_L=V>\varepsilon_g$,
$\mu_R=0<\varepsilon_g$ and $T\to0$:

| process | rate | plateau value |
|---|---|---|
| left **inject** $0\to1$  | $\gamma_L|W_L|^2\,f_L(\varepsilon_g)$       | $\gamma_L|W_L|^2$ |
| left extract $1\to0$     | $\gamma_L|W_L|^2\,[1-f_L(\varepsilon_g)]$   | $0$ |
| right inject $0\to1$     | $\gamma_R|W_R|^2\,f_R(\varepsilon_g)$       | $0$ |
| right **extract** $1\to0$| $\gamma_R|W_R|^2\,[1-f_R(\varepsilon_g)]$   | $\gamma_R|W_R|^2$ |

Only **inject-left** $\Gamma_{0\to1}=\gamma_L|W_L|^2$ and **extract-right**
$\Gamma_{1\to0}=\gamma_R|W_R|^2$ remain, with
$|W_L|^2=|\langle \tilde g,0|d_L^\dagger\hat D(\lambda_L)|0,0\rangle|^2=0.281$ and
$|W_R|^2=|\langle 0,0|d_R\hat D(\lambda_R)|\tilde g,0\rangle|^2=0.478$ — the
$\Gamma_L[|0,0\rangle\!\to\!|\tilde g,0\rangle]/\gamma_L$ and $\Gamma_R[|\tilde g,0\rangle\!\to\!|0,0\rangle]/\gamma_R$ entries of the rate matrices.

**3. Steady state.** With $P_0+P_1=1$ and detailed balance $P_0\,\Gamma_{0\to1}=P_1\,\Gamma_{1\to0}$,
$$P_0=\frac{\Gamma_{1\to0}}{\Gamma_{0\to1}+\Gamma_{1\to0}},\qquad
P_1=\frac{\Gamma_{0\to1}}{\Gamma_{0\to1}+\Gamma_{1\to0}}.$$

**4. Current.** Through the left junction only injection flows ($f_L=1$ kills the back-term):
$$I_{\rm phys}=\Gamma_{0\to1}\,P_0=\frac{\Gamma_{0\to1}\,\Gamma_{1\to0}}{\Gamma_{0\to1}+\Gamma_{1\to0}}
=\frac{(\gamma_L|W_L|^2)(\gamma_R|W_R|^2)}{\gamma_L|W_L|^2+\gamma_R|W_R|^2},$$
the series combination of the in/out rates — the slower one bottlenecks.

**5. Normalisation → harmonic mean.** `TransportCalculator.compute_current` divides by
$\text{norm}=\gamma_L\gamma_R/(\gamma_L+\gamma_R)$ (cell 30), so
$$I_1=\frac{I_{\rm phys}}{\text{norm}}=\frac{(\gamma_L+\gamma_R)\,|W_L|^2|W_R|^2}{\gamma_L|W_L|^2+\gamma_R|W_R|^2}
\;\xrightarrow{\ \gamma_L=\gamma_R\ }\;\frac{2\,|W_L|^2|W_R|^2}{|W_L|^2+|W_R|^2}.$$
Numerically $I_1=\dfrac{2(0.281)(0.478)}{0.281+0.478}=0.354.$

**Remarks.** (i) Symmetric in $R_{\rm inj}\leftrightarrow R_{\rm ext}$ — a series circuit doesn't care
which leg is which. (ii) Dominated by the **smaller** rate: for large $\lambda_L$,
$R_{\rm inj}=\tfrac12 e^{-\lambda_L^2}\times(\dots)\to$ small and $I_1\to 2R_{\rm inj}\propto
e^{-\lambda_L^2}$. (iii) The normalisation makes the *unsuppressed* plateau
($|W_L|^2=|W_R|^2=\tfrac12$) read $I_1=0.5$. (iv) This two-state reduction is exact **only** on the
first plateau; above $\approx0.73$ more states enter and the full `RateEquationSolver` (used by the
I–V curve) is required.

### 12 — the polaron form factor (definition)

The "polaron form factor" quoted above is the correction that turns the idealized Franck–Condon
factor $e^{-\lambda_\ell^2/2}$ into the *actual* dressed matrix element. The dressed elastic amplitude
factorizes as

$$
W_\ell=\langle \tilde g,0|\,d_\ell^\dagger\hat D(\lambda_\ell)\,|0,0\rangle
=\underbrace{e^{-\lambda_\ell^2/2}}_{\text{bare FC factor}}\;
\underbrace{\frac{1}{\sqrt2}\sum_m \frac{\lambda_\ell^m}{\sqrt{m!}}\,(b_m\pm a_m)}_{\displaystyle F_\ell\;=\;\text{polaron form factor}},
$$

where $|\tilde g,0\rangle=\sum_m(b_m|g,m\rangle+a_m|e,m\rangle)$ is the **polaron eigenstate** ($b_0\approx0.978$
dominant, small admixtures $b_{m\ge1},a_{m\ge1}$ from `diagonalize()`; full decomposition in §6.1), $c_m=\langle m|\hat D|0\rangle
=e^{-\lambda_\ell^2/2}\lambda_\ell^m/\sqrt{m!}$, and the sign is **$+$ for the left lead (dot 1)**,
**$-$ for the right (dot 2)** since $d_{1,2}^\dagger=\tfrac1{\sqrt2}(d_g^\dagger\pm d_e^\dagger)$.

- **Pure-product limit** ($|\tilde g,0\rangle=|g\rangle\otimes|0\rangle$, i.e. $b_0=1$, rest 0): $F_\ell=\tfrac1{\sqrt2}$
  is constant, so $|W_\ell|^2=\tfrac12 e^{-\lambda_\ell^2}$ — the clean FC factor, exactly.
- **Polaron** (inter-dot $\lambda\neq0$): the admixtures make $F_\ell$ depend on $\lambda_\ell$ **and** on the
  $\pm$ sign, so the elastic coupling deviates from the clean value and becomes left/right-asymmetric.

As a **rate** factor relative to the product value $\tfrac12 e^{-\lambda_\ell^2}$,
$\mathcal F_\ell=|W_\ell|^2/(\tfrac12 e^{-\lambda_\ell^2})$, at $\lambda=1$:
$\mathcal F_L=0.281/0.184=\mathbf{1.53}$ (left, constructive $b_m+a_m$),
$\mathcal F_R=0.123/0.184=\mathbf{0.67}$ (right, destructive $b_m-a_m$).

They are nearly **reciprocal**, $\mathcal F_L\mathcal F_R\approx1$, because
$\big[\sum_m c_m(b_m+a_m)\big]\big[\sum_m c_m(b_m-a_m)\big]=\big(\sum_m c_m b_m\big)^2-\big(\sum_m c_m a_m\big)^2\approx b_0^2$.
Hence $\sqrt{\Gamma_L\Gamma_R}$ cancels the form factor and returns the bare $e^{-\lambda^2}$ — exactly the
"geometric mean recovers $e^{-1}$" result of the `fc_factor_check` cell.